In [1]:
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\Administrator\RepurposeAlpha")
exec_dir = PROJECT_ROOT / "src" / "executive"
exec_dir.mkdir(parents=True, exist_ok=True)
(exec_dir / "__init__.py").touch()

print(f"✅ Created: {exec_dir}")

✅ Created: C:\Users\Administrator\RepurposeAlpha\src\executive


In [2]:
%%writefile C:\Users\Administrator\RepurposeAlpha\src\executive\__init__.py
"""
RepurposeAlpha — Executive Summary engine.

Aggregates results from all seven features into a single business-facing view:
- Overall verdict (RECOMMENDED / CAUTION / NOT RECOMMENDED)
- Top 3-5 candidates with strengths and risks
- Critical warnings (prioritized)
- Recommended next steps
"""
import pandas as pd
import numpy as np
from dataclasses import dataclass, field
from typing import List


@dataclass
class CandidateVerdict:
    name: str
    phase: str
    weight: float
    expected_return: float
    strengths: List[str] = field(default_factory=list)
    risks: List[str] = field(default_factory=list)


@dataclass
class Warning:
    level: str      # "critical" | "high" | "medium" | "info"
    icon: str
    text: str


@dataclass
class ExecutiveSummary:
    disease: str
    n_candidates: int
    verdict: str             # RECOMMENDED | CAUTION | NOT_RECOMMENDED
    verdict_color: str       # green | yellow | red
    one_liner: str
    top_candidates: List[CandidateVerdict]
    warnings: List[Warning]
    next_steps: List[str]
    key_metrics: dict
    confidence: str          # high | medium | low


def build_summary(disease, candidates, corr, est, w_sharpe,
                   fto_df=None, priors_df=None, synergy_df=None,
                   real_options_result=None, pricing_df=None,
                   trial_designs=None):
    """Build the executive summary from all feature outputs."""
    drug_names = dict(zip(candidates["chembl_id"], candidates["drug_name"]))
    n = len(candidates)

    # --- Warnings collection ---
    warnings = []

    # Real options
    ro_verdict = "UNKNOWN"
    if real_options_result is not None:
        if real_options_result.option_value > 0:
            ro_verdict = "PROCEED"
        else:
            ro_verdict = "DO_NOT_START"
            warnings.append(Warning(
                "high", "⏳",
                f"Real Options: DO NOT START — DCF is ${real_options_result.asset_value:.0f}M. "
                f"Even with flexibility, the project doesn't clear its cost of capital."
            ))

    # FTO
    fto_high = 0
    if fto_df is not None and not fto_df.empty:
        fto_high = int((fto_df["fto_risk"] == "high").sum())
        if fto_high >= max(1, n // 2):
            warnings.append(Warning(
                "critical", "⚖️",
                f"FTO: {fto_high}/{n} candidates are generic — no compound patent, "
                f"limited commercial protection."
            ))
        elif fto_high > 0:
            warnings.append(Warning(
                "medium", "⚖️",
                f"FTO: {fto_high} candidate(s) are generic."
            ))

    # Prior failures
    prior_high = 0
    if priors_df is not None and not priors_df.empty:
        prior_high = int((priors_df["risk_flag"] == "high").sum())
        if prior_high > 0:
            warnings.append(Warning(
                "high", "📉",
                f"Prior Results: {prior_high} candidate(s) have strong prior failures "
                f"(futility or safety)."
            ))

    # Pricing
    not_viable = 0
    if pricing_df is not None and not pricing_df.empty:
        not_viable = int((pricing_df["viability"] == "not_viable").sum())
        if not_viable > 0:
            warnings.append(Warning(
                "medium", "💰",
                f"Pricing: {not_viable} candidate(s) may not be commercially viable "
                f"at reference prices."
            ))

    # --- Overall verdict ---
    critical = sum(1 for w in warnings if w.level == "critical")
    high = sum(1 for w in warnings if w.level == "high")
    medium = sum(1 for w in warnings if w.level == "medium")

    if critical > 0 or high >= 2:
        verdict = "NOT RECOMMENDED"
        verdict_color = "red"
        one_liner = "Multiple critical issues — review warnings before proceeding."
    elif high == 1 or medium >= 2:
        verdict = "CAUTION"
        verdict_color = "yellow"
        one_liner = "Proceed with caution — one or more significant risk factors present."
    else:
        verdict = "RECOMMENDED"
        verdict_color = "green"
        one_liner = "Portfolio is well-positioned. Proceed with validation plan."

    # --- Top candidates ---
    # Sort by Max Sharpe weight
    top = w_sharpe.sort_values(ascending=False)
    top_ids = [i for i in top.index if top[i] > 0.02][:5]

    top_candidates = []
    for cid in top_ids:
        name = drug_names.get(cid, cid)
        weight = float(top[cid])
        phase_key = est.loc[cid, "phase_key"] if cid in est.index else "unknown"
        phase = phase_key.replace("_", " ").title()
        mu = float(est.loc[cid, "mu"]) if cid in est.index else 0.0

        strengths = []
        risks = []

        # Strength: high weight
        if weight > 0.15:
            strengths.append(f"High portfolio priority (weight {weight:.2f})")
        # Strength: high FTO score
        if fto_df is not None and not fto_df.empty:
            row = fto_df[fto_df["chembl_id"] == cid]
            if not row.empty:
                score = float(row.iloc[0]["fto_score"])
                if score >= 0.65:
                    strengths.append("Strong IP position")
                elif score < 0.30:
                    risks.append("Generic — no commercial protection")
        # Risk: correlated peer
        correlated = [(drug_names.get(j, j), float(corr.loc[cid, j]))
                      for j in corr.columns if j != cid and corr.loc[cid, j] > 0.4]
        if correlated:
            peer, r = max(correlated, key=lambda x: x[1])
            risks.append(f"Correlated with {peer} (r={r:.2f}) — may be redundant")
        # Risk: prior failures
        if priors_df is not None and not priors_df.empty:
            row = priors_df[priors_df["chembl_id"] == cid]
            if not row.empty and row.iloc[0]["risk_flag"] in ("high", "medium"):
                risks.append(f"Prior failure signal ({row.iloc[0]['risk_flag']})")

        top_candidates.append(CandidateVerdict(
            name=name, phase=phase, weight=weight,
            expected_return=mu, strengths=strengths, risks=risks,
        ))

    # --- Next steps ---
    next_steps = []
    if top_candidates:
        names = [c.name for c in top_candidates[:3]]
        next_steps.append(
            f"Focus validation budget on top {len(names)} candidates: "
            f"{', '.join(names)}"
        )
    if priors_df is not None and prior_high > 0:
        next_steps.append(
            "Review prior failure details before committing to candidates with red flags"
        )
    if fto_df is not None and fto_high > 0:
        next_steps.append(
            "Confirm IP strategy for generic candidates — consider formulation or "
            "method-of-use patents"
        )
    if trial_designs:
        cheapest = min(trial_designs, key=lambda d: d.expected_cost_usd)
        baseline = max(trial_designs, key=lambda d: d.expected_cost_usd)
        savings = baseline.expected_cost_usd - cheapest.expected_cost_usd
        next_steps.append(
            f"Consider {cheapest.name.split('(')[0].strip()} — saves "
            f"${savings/1e6:.1f}M vs. independent trials"
        )
    if not next_steps:
        next_steps.append("Analysis complete — review each tab for details")

    # --- Key metrics ---
    key_metrics = {
        "Portfolio Sharpe": f"{0.0:.2f}",  # placeholder, filled by app
        "Candidates": str(n),
        "Real Options": ro_verdict.replace("_", " "),
        "FTO Status": f"{n - fto_high}/{n} protected" if fto_df is not None else "N/A",
        "Prior Flags": f"{prior_high} high" if priors_df is not None else "N/A",
    }

    # --- Confidence ---
    if n >= 8 and fto_df is not None and priors_df is not None:
        confidence = "high"
    elif n >= 4:
        confidence = "medium"
    else:
        confidence = "low"

    return ExecutiveSummary(
        disease=disease,
        n_candidates=n,
        verdict=verdict,
        verdict_color=verdict_color,
        one_liner=one_liner,
        top_candidates=top_candidates,
        warnings=warnings,
        next_steps=next_steps,
        key_metrics=key_metrics,
        confidence=confidence,
    )

Overwriting C:\Users\Administrator\RepurposeAlpha\src\executive\__init__.py


In [3]:
%%writefile C:\Users\Administrator\RepurposeAlpha\app\app.py
"""
RepurposeAlpha — Streamlit app v0.3.0 (organized).
Seven top-level tabs. Executive Summary first.
"""
import sys, io
from pathlib import Path

APP_DIR = Path(__file__).resolve().parent
ROOT    = APP_DIR.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(APP_DIR))

import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pypfopt import EfficientFrontier

import auth, assumptions, discovery, correlation, cache as cache_mod
import returns, reporting, options, fto, trials, crosstarget
import negativedata, synergy, pricing, executive

st.set_page_config(page_title="RepurposeAlpha", page_icon="🧬", layout="wide")

# ---------- Auth ----------
user = auth.require_login(min_role="viewer")

# ---------- Header ----------
col_title, col_user = st.columns([5, 1])
with col_title:
    st.title("🧬 RepurposeAlpha")
    st.caption("Portfolio decision support for drug repurposing")
with col_user:
    st.caption(f"👤 {user['username']} ({user['role']})")
    if st.button("Log out"):
        auth.log_event(user["username"], "logout")
        st.session_state["user"] = None
        st.rerun()

# ============================================================
# QUERY BAR
# ============================================================
with st.container():
    q1, q2, q3, q4, q5 = st.columns([3, 1, 1, 1, 1])
    with q1:
        disease_input = st.text_input("Disease", value=st.session_state.get("disease", "Prader-Willi syndrome"))
    with q2:
        min_phase = st.selectbox("Min phase", [1, 2, 3], index=1)
    with q3:
        max_candidates = st.number_input("Max", 5, 30, 10, step=5)
    with q4:
        use_cache = st.checkbox("Cache", value=True)
    with q5:
        st.write("")
        run_analysis = st.button("🚀 Analyze", type="primary")

if disease_input:
    st.session_state["disease"] = disease_input

if run_analysis and disease_input:
    if use_cache and cache_mod.has_cache(disease_input):
        candidates, corr, meta = cache_mod.load_analysis(disease_input)
        st.success(f"✓ Loaded from cache ({meta.get('n_candidates', '?')} candidates)")
    else:
        with st.status(f"Analyzing {disease_input}...", expanded=True) as status:
            st.write("1/3 Discovering candidates from Open Targets...")
            try:
                candidates = discovery.get_candidates_for_disease(disease_input, min_phase=min_phase, max_results=max_candidates)
            except Exception as e:
                status.update(label=f"❌ Discovery failed: {e}", state="error"); st.stop()
            if candidates is None or candidates.empty:
                status.update(label=f"❌ No candidates found", state="error"); st.stop()
            st.write(f"   ✓ {len(candidates)} candidates")

            st.write("2/3 Computing correlation matrix...")
            try:
                corr = correlation.build_correlation_matrix(candidates["chembl_id"].tolist(), verbose=False)
            except Exception as e:
                status.update(label=f"❌ Correlation failed: {e}", state="error"); st.stop()
            if corr is None or corr.empty:
                status.update(label="❌ Empty matrix", state="error"); st.stop()
            st.write(f"   ✓ {len(corr)}×{len(corr)} matrix")

            st.write("3/3 Caching...")
            cache_mod.save_analysis(disease_input, candidates, corr, metadata={"min_phase": min_phase})
            status.update(label=f"✅ Complete", state="complete")

    st.session_state["candidates"] = candidates
    st.session_state["corr"] = corr
    st.session_state["disease"] = disease_input
    # Reset downstream caches on new analysis
    for key in ["fto_df", "nd_df", "syn_df", "pr_df"]:
        st.session_state.pop(key, None)

candidates = st.session_state.get("candidates")
corr       = st.session_state.get("corr")
disease    = st.session_state.get("disease", "(none)")

if candidates is None or corr is None:
    st.info("👆 Enter a disease name and click **Analyze** to begin.")
    st.stop()

# ---------- Compute base analyses ----------
tickers = list(corr.columns)
drug_names = dict(zip(candidates["chembl_id"], candidates["drug_name"]))

est = returns.estimate_returns(candidates, corr, verbose=False)
est = est.reindex(tickers).dropna(subset=["mu", "sigma"])
mu_s = est["mu"]
sigma_s = est["sigma"]

rfr_entry = assumptions.load_registry().get("risk_free_rate", {})
rfr_value = float(rfr_entry.get("value", 0.03))

cov = pd.DataFrame(np.outer(sigma_s, sigma_s) * corr.loc[est.index, est.index].values,
                   index=est.index, columns=est.index)

def solve(mode):
    ef = EfficientFrontier(mu_s, cov, weight_bounds=(0, 1))
    try:
        if mode == "sharpe": ef.max_sharpe(risk_free_rate=rfr_value)
        else: ef.min_volatility()
        w = ef.clean_weights()
        ret, vol, sharpe = ef.portfolio_performance(risk_free_rate=rfr_value)
        return pd.Series(w), ret, vol, sharpe
    except Exception:
        return pd.Series(0.0, index=est.index), 0.0, 0.0, 0.0

w_sharpe, r_s, v_s, s_s = solve("sharpe")
w_minvol, r_m, v_m, _   = solve("minvol")

# ---------- Sidebar ----------
st.sidebar.header("Context")
st.sidebar.markdown(f"**Disease:** {disease}")
st.sidebar.markdown(f"**Candidates:** {len(est)}")
st.sidebar.markdown(f"**Risk-free rate:** {rfr_value:.4f}")
st.sidebar.caption("Source: FRED DGS10 (live)")

# ============================================================
# MAIN TABS (7 top-level)
# ============================================================
tab_names = ["📊 Executive Summary", "📈 Portfolio", "🧬 Strategic",
             "📚 Evidence", "⚙️ Assumptions", "📄 Report"]
if auth.has_role(user, "admin"):
    tab_names.append("🛡️ Admin")
tabs = st.tabs(tab_names)


# ---------- TAB 1: EXECUTIVE SUMMARY ----------
with tabs[0]:
    st.subheader(f"Executive Summary — {disease.title()}")

    # Run all analyses silently
    with st.spinner("Aggregating feature outputs..."):
        try:
            fto_df = fto.assess_portfolio(candidates, verbose=False)
        except Exception: fto_df = None
        try:
            priors_df = negativedata.analyze_portfolio(candidates, disease, verbose=False)
        except Exception: priors_df = None
        try:
            synergy_df = synergy.find_synergy_pairs(candidates, corr, top_n=3, check_trials=False, verbose=False)
        except Exception: synergy_df = None
        try:
            pricing_df = pricing.analyze_portfolio(candidates, disease, verbose=False)
        except Exception: pricing_df = None
        try:
            _phases_ro = [
                options.Phase("Phase I", 2.0, float(assumptions.get_value("cost.phase_1_usd", 10e6)), float(assumptions.get_value("pos.phase_1", 0.63))),
                options.Phase("Phase II", 2.0, float(assumptions.get_value("cost.phase_2_usd", 25e6)), float(assumptions.get_value("pos.phase_2", 0.31))),
                options.Phase("Phase III", 3.0, float(assumptions.get_value("cost.phase_3_usd", 100e6)), float(assumptions.get_value("pos.phase_3", 0.58))),
                options.Phase("NDA", 1.0, 5e6, float(assumptions.get_value("pos.nda_bla", 0.85))),
            ]
            ro_result = options.RealOptionsEngine(phases=_phases_ro, peak_sales=200e6, volatility=0.40,
                                                  risk_free_rate=rfr_value, discount_rate=0.10).value()
        except Exception: ro_result = None
        try:
            trial_designs = trials.recommend_designs(n_candidates=min(len(est), 6))
        except Exception: trial_designs = None

        summary = executive.build_summary(
            disease=disease, candidates=candidates, corr=corr, est=est,
            w_sharpe=w_sharpe, fto_df=fto_df, priors_df=priors_df,
            synergy_df=synergy_df, real_options_result=ro_result,
            pricing_df=pricing_df, trial_designs=trial_designs,
        )

    # ---- VERDICT BANNER ----
    color_map = {"green": "#10b981", "yellow": "#f59e0b", "red": "#ef4444"}
    verdict_color = color_map[summary.verdict_color]
    st.markdown(
        f"""<div style="background:{verdict_color}22; border-left:5px solid {verdict_color};
        padding:20px 24px; border-radius:8px; margin-bottom:24px;">
        <div style="font-size:12px; color:#94a3b8; letter-spacing:1.5px; text-transform:uppercase;">Overall verdict</div>
        <div style="font-size:32px; font-weight:700; color:{verdict_color}; margin:4px 0 8px 0;">
            {summary.verdict}</div>
        <div style="font-size:15px; color:#cbd5e1;">{summary.one_liner}</div>
        </div>""", unsafe_allow_html=True)

    # ---- KEY METRICS ----
    mc1, mc2, mc3, mc4, mc5 = st.columns(5)
    mc1.metric("Portfolio Sharpe", f"{s_s:.2f}")
    mc2.metric("Candidates", len(est))
    mc3.metric("Real Options", summary.key_metrics.get("Real Options", "—"))
    mc4.metric("FTO", summary.key_metrics.get("FTO Status", "—"))
    mc5.metric("Confidence", summary.confidence.title())

    st.divider()

    # ---- TOP CANDIDATES ----
    st.markdown("### 🎯 Top candidates")
    if summary.top_candidates:
        cols = st.columns(min(len(summary.top_candidates), 3))
        for i, c in enumerate(summary.top_candidates[:3]):
            with cols[i]:
                st.markdown(f"**{c.name}**")
                st.caption(f"{c.phase} · weight {c.weight:.3f} · est. return {c.expected_return:.3f}")
                for s in c.strengths[:2]:
                    st.markdown(f"✅ {s}")
                for r in c.risks[:2]:
                    st.markdown(f"⚠️ {r}")
    else:
        st.info("No candidates passed the portfolio threshold.")

    st.divider()

    # ---- WARNINGS ----
    if summary.warnings:
        st.markdown("### ⚠️ Critical items to review")
        for w in summary.warnings:
            if w.level == "critical":
                st.error(f"{w.icon} {w.text}")
            elif w.level == "high":
                st.warning(f"{w.icon} {w.text}")
            else:
                st.info(f"{w.icon} {w.text}")

    # ---- NEXT STEPS ----
    st.markdown("### 📋 Recommended next steps")
    for i, step in enumerate(summary.next_steps, 1):
        st.markdown(f"**{i}.** {step}")

    st.divider()
    st.caption("Full details in each tab. Download the PDF from the Report tab.")


# ---------- TAB 2: PORTFOLIO ----------
with tabs[1]:
    sub = st.tabs(["🔗 Correlation", "📈 Frontier", "💼 Weights", "🎚️ Sensitivity"])

    with sub[0]:
        st.markdown("**Biological similarity between candidates.** 1.0 = same bet, 0.0 = independent.")
        labelled = corr.rename(index=drug_names, columns=drug_names)
        fig = px.imshow(labelled, text_auto=".2f", color_continuous_scale="RdYlGn_r", zmin=0, zmax=1, aspect="auto")
        fig.update_layout(height=max(400, 25 * len(tickers)))
        st.plotly_chart(fig, width="stretch")

    with sub[1]:
        st.markdown("**Risk-return tradeoff.** Each point is a random portfolio.")
        n = 3000
        rng = np.random.default_rng(42)
        rand_w = rng.dirichlet(np.ones(len(mu_s)), n)
        rets = rand_w @ mu_s.values
        vols = np.sqrt(np.einsum("ij,jk,ik->i", rand_w, cov.values, rand_w))
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=vols, y=rets, mode="markers",
                                 marker=dict(size=4, color=rets/vols, colorscale="Viridis")))
        fig.add_trace(go.Scatter(x=[v_s, v_m], y=[r_s, r_m], mode="markers+text",
                                 marker=dict(size=18, color=["gold", "silver"]),
                                 text=["Max Sharpe", "Min Vol"], textposition="top center"))
        fig.update_layout(xaxis_title="Risk", yaxis_title="Expected return", height=520)
        st.plotly_chart(fig, width="stretch")

    with sub[2]:
        c1, c2 = st.columns(2)
        with c1:
            st.markdown("**Max Sharpe**")
            df = pd.DataFrame({"weight": w_sharpe})
            df.index = [drug_names.get(i, i) for i in df.index]
            st.dataframe(df.style.format("{:.3f}"), width="stretch")
            st.metric("Return", f"{r_s:.3f}")
            st.metric("Volatility", f"{v_s:.3f}")
            st.metric("Sharpe", f"{s_s:.3f}")
        with c2:
            st.markdown("**Min Volatility**")
            df = pd.DataFrame({"weight": w_minvol})
            df.index = [drug_names.get(i, i) for i in df.index]
            st.dataframe(df.style.format("{:.3f}"), width="stretch")
            st.metric("Return", f"{r_m:.3f}")
            st.metric("Volatility", f"{v_m:.3f}")

    with sub[3]:
        if len(tickers) >= 2:
            t1 = st.selectbox("Drug A", tickers, index=0, format_func=lambda x: drug_names.get(x, x))
            t2 = st.selectbox("Drug B", tickers, index=1, format_func=lambda x: drug_names.get(x, x))
            rho = st.slider("Correlation", 0.0, 1.0, float(corr.loc[t1, t2]), 0.01)
            corr2 = corr.copy(); corr2.loc[t1, t2] = rho; corr2.loc[t2, t1] = rho
            cov2 = pd.DataFrame(np.outer(sigma_s, sigma_s) * corr2.loc[est.index, est.index].values,
                                index=est.index, columns=est.index)
            ef = EfficientFrontier(mu_s, cov2, weight_bounds=(0, 1))
            try:
                ef.max_sharpe(risk_free_rate=rfr_value)
                w = pd.Series(ef.clean_weights())
                w.index = [drug_names.get(i, i) for i in w.index]
                st.bar_chart(w)
            except Exception as e:
                st.error(f"Solver error: {e}")


# ---------- TAB 3: STRATEGIC ----------
with tabs[2]:
    sub = st.tabs(["⏳ Real Options", "⚖️ FTO", "🧪 Trials", "💰 Pricing"])

    with sub[0]:
        st.markdown("**When should you kill this project?** Each candidate is modeled as a series of phase gates.")
        c1_, c2_ = st.columns([1, 2])
        with c1_:
            peak = st.number_input("Peak sales ($M)", 50, 2000, 200, step=50) * 1e6
            vol  = st.slider("Volatility", 0.20, 0.80, 0.40, 0.05)
        _ph = [
            options.Phase("Phase I", 2.0, float(assumptions.get_value("cost.phase_1_usd", 10e6)), float(assumptions.get_value("pos.phase_1", 0.63))),
            options.Phase("Phase II", 2.0, float(assumptions.get_value("cost.phase_2_usd", 25e6)), float(assumptions.get_value("pos.phase_2", 0.31))),
            options.Phase("Phase III", 3.0, float(assumptions.get_value("cost.phase_3_usd", 100e6)), float(assumptions.get_value("pos.phase_3", 0.58))),
            options.Phase("NDA", 1.0, 5e6, float(assumptions.get_value("pos.nda_bla", 0.85))),
        ]
        r = options.RealOptionsEngine(phases=_ph, peak_sales=peak, volatility=vol,
                                      risk_free_rate=rfr_value, discount_rate=0.10).value()
        mc1, mc2, mc3 = st.columns(3)
        mc1.metric("DCF", f"${r.asset_value:.1f}M")
        mc2.metric("Option value", f"${r.option_value:.1f}M")
        mc3.metric("Flexibility premium", f"${r.abandonment_value:.1f}M")
        if r.option_value > 0:
            st.success(f"✅ PROCEED — option value ${r.option_value:.1f}M")
        else:
            st.error(f"❌ DO NOT START — DCF ${r.asset_value:.1f}M")
        gate_df = pd.DataFrame(r.decision_tree)
        st.dataframe(gate_df, width="stretch", hide_index=True)

    with sub[1]:
        st.markdown("**Commercial IP position.** Screening only — not legal advice.")
        fto_tab = fto.assess_portfolio(candidates, verbose=False)
        mc1, mc2, mc3 = st.columns(3)
        counts = fto_tab["fto_risk"].value_counts().to_dict()
        mc1.metric("🟢 Low", counts.get("low", 0))
        mc2.metric("🟡 Medium", counts.get("medium", 0))
        mc3.metric("🔴 High", counts.get("high", 0))
        disp = fto_tab[["drug_name", "status", "fto_risk", "fto_score", "rationale"]].copy()
        disp.columns = ["Drug", "Stage", "Risk", "Score", "Rationale"]
        st.dataframe(disp, width="stretch", hide_index=True)

    with sub[2]:
        st.markdown("**Trial architecture.** Platform trials share control arms — usually cheaper and faster.")
        n_test = st.slider("Candidates to test", 2, 10, min(len(est), 6))
        designs = trials.recommend_designs(n_candidates=n_test)
        df_t = trials.format_recommendation(designs)
        cheapest = min(designs, key=lambda d: d.expected_cost_usd)
        baseline = max(designs, key=lambda d: d.expected_cost_usd)
        savings = baseline.expected_cost_usd - cheapest.expected_cost_usd
        mc1, mc2, mc3 = st.columns(3)
        mc1.metric("Cheapest", cheapest.name.split("(")[0].strip(), f"${cheapest.expected_cost_usd/1e6:.1f}M")
        mc2.metric("Fastest", min(designs, key=lambda d: d.expected_duration_months).name.split("(")[0].strip())
        mc3.metric("Savings vs fixed", f"${savings/1e6:.1f}M")
        st.dataframe(df_t[["Design", "Arms", "Sample size", "Duration (mo)", "Cost ($M)", "Savings %"]],
                     width="stretch", hide_index=True)

    with sub[3]:
        st.markdown("**Sustainable price per patient per year.** Recovers development cost + WACC.")
        pr_df = pricing.analyze_portfolio(candidates, disease, verbose=False)
        counts = pr_df["viability"].value_counts().to_dict()
        mc1, mc2, mc3 = st.columns(3)
        mc1.metric("🟢 Viable", counts.get("commercially_viable", 0))
        mc2.metric("🟡 Marginal", counts.get("marginal", 0))
        mc3.metric("🔴 Not viable", counts.get("not_viable", 0))
        disp = pr_df[["drug_name", "phase", "sustainable_usd", "reference_usd", "viability"]].copy()
        disp.columns = ["Drug", "Phase", "Sustainable $/yr", "Reference $/yr", "Viability"]
        st.dataframe(disp, width="stretch", hide_index=True)


# ---------- TAB 4: EVIDENCE ----------
with tabs[3]:
    sub = st.tabs(["🌐 Cross-Disease", "📉 Prior Results", "🔬 Synergy"])

    with sub[0]:
        st.markdown("**Other diseases where each candidate is tested.** More indications = more accumulated safety data.")
        if st.button("Analyze cross-disease", key="cd_btn"):
            with st.spinner("Querying Open Targets..."):
                st.session_state["cd_result"] = crosstarget.find_cross_disease_overlaps(candidates, disease, verbose=False)
        cd_df = st.session_state.get("cd_result")
        if cd_df is not None and not cd_df.empty:
            n_over = int((cd_df["n_other_diseases"] > 0).sum())
            st.metric("Cross-disease hits", f"{n_over}/{len(cd_df)}")
            disp = cd_df[["drug_name", "n_other_diseases", "top_other_disease", "all_other_diseases"]].copy()
            disp.columns = ["Drug", "# Others", "Top", "All overlaps"]
            st.dataframe(disp, width="stretch", hide_index=True)
        else:
            st.info("Click above to run cross-disease analysis.")

    with sub[1]:
        st.markdown("**Prior terminated trials.** Weighted by failure reason (futility/safety = strong; enrollment/business = weak).")
        if st.button("Analyze prior results", key="nd_btn"):
            with st.spinner("Querying ClinicalTrials.gov..."):
                st.session_state["nd_result"] = negativedata.analyze_portfolio(candidates, disease, verbose=False)
        nd_df = st.session_state.get("nd_result")
        if nd_df is not None and not nd_df.empty:
            counts = nd_df["risk_flag"].value_counts().to_dict()
            mc1, mc2, mc3, mc4 = st.columns(4)
            mc1.metric("🔴 High", counts.get("high", 0))
            mc2.metric("🟡 Medium", counts.get("medium", 0))
            mc3.metric("🟢 Low", counts.get("low", 0))
            mc4.metric("✅ Clean", counts.get("clean", 0))
            disp = nd_df[["drug_name", "n_failed", "n_relevant", "risk_flag", "weighted_risk", "breakdown"]].copy()
            disp.columns = ["Drug", "# Failed", "# Relevant", "Risk", "Weighted", "Breakdown"]
            st.dataframe(disp, width="stretch", hide_index=True)
        else:
            st.info("Click above to check prior failures.")

    with sub[2]:
        st.markdown("**Pairs to test together.** Peak synergy at correlation 0.1–0.5 — shared context, different mechanism.")
        if st.button("Analyze synergy", key="syn_btn"):
            with st.spinner("Scoring pairs..."):
                st.session_state["syn_result"] = synergy.find_synergy_pairs(candidates, corr, top_n=15, check_trials=False, verbose=False)
        syn_df = st.session_state.get("syn_result")
        if syn_df is not None and not syn_df.empty:
            top = synergy.top_combination_recommendation(syn_df)
            if top.get("found"):
                st.success(f"🎯 Top pair: **{top['drug_a']} + {top['drug_b']}** (score {top['score']:.2f})")
            disp = syn_df[["drug_a", "drug_b", "correlation", "synergy_score", "rationale"]].copy()
            disp.columns = ["Drug A", "Drug B", "Correlation", "Synergy", "Rationale"]
            st.dataframe(disp, width="stretch", hide_index=True)
        else:
            st.info("Click above to score combinations.")


# ---------- TAB 5: ASSUMPTIONS ----------
with tabs[4]:
    st.subheader("Assumption registry")
    st.caption("Every numeric input with source, confidence, and timestamp.")
    reg = assumptions.load_registry()
    rows = []
    def _flat(node, prefix=""):
        for k, v in node.items():
            p = f"{prefix}.{k}" if prefix else k
            if isinstance(v, dict) and "value" in v:
                rows.append({"Key": p, "Value": v["value"], "Confidence": v.get("confidence", "?"),
                             "Source": v.get("source", "")})
            elif isinstance(v, dict):
                _flat(v, p)
    _flat(reg)
    st.dataframe(pd.DataFrame(rows), width="stretch", hide_index=True)


# ---------- TAB 6: REPORT ----------
with tabs[5]:
    st.subheader("📄 Download report")
    st.markdown("Generate a 6-page PDF with methodology, charts, and full provenance.")
    if st.button("📥 Generate PDF report", type="primary", key="pdf_btn"):
        with st.spinner("Generating..."):
            import tempfile
            with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmp:
                tmp_path = Path(tmp.name)
            findings = f"Analysis of {disease} with {len(est)} candidates. Portfolio Sharpe: {s_s:.2f}."
            reporting.build_report(
                path=tmp_path, title=f"RepurposeAlpha — {disease.title()}",
                user=user["username"], disease=disease,
                corr=corr.rename(index=drug_names, columns=drug_names),
                mu=mu_s, sigma=sigma_s, cov=cov,
                w_sharpe=w_sharpe.rename(index=drug_names),
                w_minvol=w_minvol.rename(index=drug_names),
                registry=assumptions.load_registry(), findings=findings,
            )
            st.session_state["pdf_bytes"] = tmp_path.read_bytes()
            st.session_state["pdf_name"] = f"RepurposeAlpha_{disease.replace(' ', '_')}.pdf"
            tmp_path.unlink()
            st.success(f"✅ Ready ({len(st.session_state['pdf_bytes']):,} bytes)")
    if "pdf_bytes" in st.session_state:
        st.download_button("⬇️  Download PDF", data=st.session_state["pdf_bytes"],
                           file_name=st.session_state.get("pdf_name", "report.pdf"),
                           mime="application/pdf", type="primary")


# ---------- TAB 7: ADMIN ----------
if auth.has_role(user, "admin") and len(tabs) == 7:
    with tabs[6]:
        st.subheader("🛡️ Admin panel")
        st.markdown("**Users**")
        st.dataframe(pd.DataFrame(auth.list_users(), columns=["username", "role", "created_at"]),
                     width="stretch", hide_index=True)
        st.markdown("**Audit log**")
        st.dataframe(pd.DataFrame(auth.list_audit(50),
                                  columns=["timestamp", "user", "action", "detail"]),
                     width="stretch", hide_index=True)


st.divider()
st.caption(f"RepurposeAlpha v0.3.0 · {disease} · {len(est)} candidates")

Overwriting C:\Users\Administrator\RepurposeAlpha\app\app.py


In [4]:
%%writefile C:\Users\Administrator\RepurposeAlpha\app\app.py
"""
RepurposeAlpha — Streamlit app v0.3.0 (organized).
Seven top-level tabs. Executive Summary first.
"""
import sys, io
from pathlib import Path

APP_DIR = Path(__file__).resolve().parent
ROOT    = APP_DIR.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(APP_DIR))

import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pypfopt import EfficientFrontier

import auth, assumptions, discovery, correlation, cache as cache_mod
import returns, reporting, options, fto, trials, crosstarget
import negativedata, synergy, pricing, executive

st.set_page_config(page_title="RepurposeAlpha", page_icon="🧬", layout="wide")

# ---------- Auth ----------
user = auth.require_login(min_role="viewer")

# ---------- Header ----------
col_title, col_user = st.columns([5, 1])
with col_title:
    st.title("🧬 RepurposeAlpha")
    st.caption("Portfolio decision support for drug repurposing")
with col_user:
    st.caption(f"👤 {user['username']} ({user['role']})")
    if st.button("Log out"):
        auth.log_event(user["username"], "logout")
        st.session_state["user"] = None
        st.rerun()

# ============================================================
# QUERY BAR
# ============================================================
with st.container():
    q1, q2, q3, q4, q5 = st.columns([3, 1, 1, 1, 1])
    with q1:
        disease_input = st.text_input("Disease", value=st.session_state.get("disease", "Prader-Willi syndrome"))
    with q2:
        min_phase = st.selectbox("Min phase", [1, 2, 3], index=1)
    with q3:
        max_candidates = st.number_input("Max", 5, 30, 10, step=5)
    with q4:
        use_cache = st.checkbox("Cache", value=True)
    with q5:
        st.write("")
        run_analysis = st.button("🚀 Analyze", type="primary")

if disease_input:
    st.session_state["disease"] = disease_input

if run_analysis and disease_input:
    if use_cache and cache_mod.has_cache(disease_input):
        candidates, corr, meta = cache_mod.load_analysis(disease_input)
        st.success(f"✓ Loaded from cache ({meta.get('n_candidates', '?')} candidates)")
    else:
        with st.status(f"Analyzing {disease_input}...", expanded=True) as status:
            st.write("1/3 Discovering candidates from Open Targets...")
            try:
                candidates = discovery.get_candidates_for_disease(disease_input, min_phase=min_phase, max_results=max_candidates)
            except Exception as e:
                status.update(label=f"❌ Discovery failed: {e}", state="error"); st.stop()
            if candidates is None or candidates.empty:
                status.update(label=f"❌ No candidates found", state="error"); st.stop()
            st.write(f"   ✓ {len(candidates)} candidates")

            st.write("2/3 Computing correlation matrix...")
            try:
                corr = correlation.build_correlation_matrix(candidates["chembl_id"].tolist(), verbose=False)
            except Exception as e:
                status.update(label=f"❌ Correlation failed: {e}", state="error"); st.stop()
            if corr is None or corr.empty:
                status.update(label="❌ Empty matrix", state="error"); st.stop()
            st.write(f"   ✓ {len(corr)}×{len(corr)} matrix")

            st.write("3/3 Caching...")
            cache_mod.save_analysis(disease_input, candidates, corr, metadata={"min_phase": min_phase})
            status.update(label=f"✅ Complete", state="complete")

    st.session_state["candidates"] = candidates
    st.session_state["corr"] = corr
    st.session_state["disease"] = disease_input
    # Reset downstream caches on new analysis
    for key in ["fto_df", "nd_df", "syn_df", "pr_df"]:
        st.session_state.pop(key, None)

candidates = st.session_state.get("candidates")
corr       = st.session_state.get("corr")
disease    = st.session_state.get("disease", "(none)")

if candidates is None or corr is None:
    st.info("👆 Enter a disease name and click **Analyze** to begin.")
    st.stop()

# ---------- Compute base analyses ----------
tickers = list(corr.columns)
drug_names = dict(zip(candidates["chembl_id"], candidates["drug_name"]))

est = returns.estimate_returns(candidates, corr, verbose=False)
est = est.reindex(tickers).dropna(subset=["mu", "sigma"])
mu_s = est["mu"]
sigma_s = est["sigma"]

rfr_entry = assumptions.load_registry().get("risk_free_rate", {})
rfr_value = float(rfr_entry.get("value", 0.03))

cov = pd.DataFrame(np.outer(sigma_s, sigma_s) * corr.loc[est.index, est.index].values,
                   index=est.index, columns=est.index)

def solve(mode):
    ef = EfficientFrontier(mu_s, cov, weight_bounds=(0, 1))
    try:
        if mode == "sharpe": ef.max_sharpe(risk_free_rate=rfr_value)
        else: ef.min_volatility()
        w = ef.clean_weights()
        ret, vol, sharpe = ef.portfolio_performance(risk_free_rate=rfr_value)
        return pd.Series(w), ret, vol, sharpe
    except Exception:
        return pd.Series(0.0, index=est.index), 0.0, 0.0, 0.0

w_sharpe, r_s, v_s, s_s = solve("sharpe")
w_minvol, r_m, v_m, _   = solve("minvol")

# ---------- Sidebar ----------
st.sidebar.header("Context")
st.sidebar.markdown(f"**Disease:** {disease}")
st.sidebar.markdown(f"**Candidates:** {len(est)}")
st.sidebar.markdown(f"**Risk-free rate:** {rfr_value:.4f}")
st.sidebar.caption("Source: FRED DGS10 (live)")

# ============================================================
# MAIN TABS (7 top-level)
# ============================================================
tab_names = ["📊 Executive Summary", "📈 Portfolio", "🧬 Strategic",
             "📚 Evidence", "⚙️ Assumptions", "📄 Report"]
if auth.has_role(user, "admin"):
    tab_names.append("🛡️ Admin")
tabs = st.tabs(tab_names)


# ---------- TAB 1: EXECUTIVE SUMMARY ----------
with tabs[0]:
    st.subheader(f"Executive Summary — {disease.title()}")

    # Run all analyses silently
    with st.spinner("Aggregating feature outputs..."):
        try:
            fto_df = fto.assess_portfolio(candidates, verbose=False)
        except Exception: fto_df = None
        try:
            priors_df = negativedata.analyze_portfolio(candidates, disease, verbose=False)
        except Exception: priors_df = None
        try:
            synergy_df = synergy.find_synergy_pairs(candidates, corr, top_n=3, check_trials=False, verbose=False)
        except Exception: synergy_df = None
        try:
            pricing_df = pricing.analyze_portfolio(candidates, disease, verbose=False)
        except Exception: pricing_df = None
        try:
            _phases_ro = [
                options.Phase("Phase I", 2.0, float(assumptions.get_value("cost.phase_1_usd", 10e6)), float(assumptions.get_value("pos.phase_1", 0.63))),
                options.Phase("Phase II", 2.0, float(assumptions.get_value("cost.phase_2_usd", 25e6)), float(assumptions.get_value("pos.phase_2", 0.31))),
                options.Phase("Phase III", 3.0, float(assumptions.get_value("cost.phase_3_usd", 100e6)), float(assumptions.get_value("pos.phase_3", 0.58))),
                options.Phase("NDA", 1.0, 5e6, float(assumptions.get_value("pos.nda_bla", 0.85))),
            ]
            ro_result = options.RealOptionsEngine(phases=_phases_ro, peak_sales=200e6, volatility=0.40,
                                                  risk_free_rate=rfr_value, discount_rate=0.10).value()
        except Exception: ro_result = None
        try:
            trial_designs = trials.recommend_designs(n_candidates=min(len(est), 6))
        except Exception: trial_designs = None

        summary = executive.build_summary(
            disease=disease, candidates=candidates, corr=corr, est=est,
            w_sharpe=w_sharpe, fto_df=fto_df, priors_df=priors_df,
            synergy_df=synergy_df, real_options_result=ro_result,
            pricing_df=pricing_df, trial_designs=trial_designs,
        )

    # ---- VERDICT BANNER ----
    color_map = {"green": "#10b981", "yellow": "#f59e0b", "red": "#ef4444"}
    verdict_color = color_map[summary.verdict_color]
    st.markdown(
        f"""<div style="background:{verdict_color}22; border-left:5px solid {verdict_color};
        padding:20px 24px; border-radius:8px; margin-bottom:24px;">
        <div style="font-size:12px; color:#94a3b8; letter-spacing:1.5px; text-transform:uppercase;">Overall verdict</div>
        <div style="font-size:32px; font-weight:700; color:{verdict_color}; margin:4px 0 8px 0;">
            {summary.verdict}</div>
        <div style="font-size:15px; color:#cbd5e1;">{summary.one_liner}</div>
        </div>""", unsafe_allow_html=True)

    # ---- KEY METRICS ----
    mc1, mc2, mc3, mc4, mc5 = st.columns(5)
    mc1.metric("Portfolio Sharpe", f"{s_s:.2f}")
    mc2.metric("Candidates", len(est))
    mc3.metric("Real Options", summary.key_metrics.get("Real Options", "—"))
    mc4.metric("FTO", summary.key_metrics.get("FTO Status", "—"))
    mc5.metric("Confidence", summary.confidence.title())

    st.divider()

    # ---- TOP CANDIDATES ----
    st.markdown("### 🎯 Top candidates")
    if summary.top_candidates:
        cols = st.columns(min(len(summary.top_candidates), 3))
        for i, c in enumerate(summary.top_candidates[:3]):
            with cols[i]:
                st.markdown(f"**{c.name}**")
                st.caption(f"{c.phase} · weight {c.weight:.3f} · est. return {c.expected_return:.3f}")
                for s in c.strengths[:2]:
                    st.markdown(f"✅ {s}")
                for r in c.risks[:2]:
                    st.markdown(f"⚠️ {r}")
    else:
        st.info("No candidates passed the portfolio threshold.")

    st.divider()

    # ---- WARNINGS ----
    if summary.warnings:
        st.markdown("### ⚠️ Critical items to review")
        for w in summary.warnings:
            if w.level == "critical":
                st.error(f"{w.icon} {w.text}")
            elif w.level == "high":
                st.warning(f"{w.icon} {w.text}")
            else:
                st.info(f"{w.icon} {w.text}")

    # ---- NEXT STEPS ----
    st.markdown("### 📋 Recommended next steps")
    for i, step in enumerate(summary.next_steps, 1):
        st.markdown(f"**{i}.** {step}")

    st.divider()
    st.caption("Full details in each tab. Download the PDF from the Report tab.")


# ---------- TAB 2: PORTFOLIO ----------
with tabs[1]:
    sub = st.tabs(["🔗 Correlation", "📈 Frontier", "💼 Weights", "🎚️ Sensitivity"])

    with sub[0]:
        st.markdown("**Biological similarity between candidates.** 1.0 = same bet, 0.0 = independent.")
        labelled = corr.rename(index=drug_names, columns=drug_names)
        fig = px.imshow(labelled, text_auto=".2f", color_continuous_scale="RdYlGn_r", zmin=0, zmax=1, aspect="auto")
        fig.update_layout(height=max(400, 25 * len(tickers)))
        st.plotly_chart(fig, width="stretch")

    with sub[1]:
        st.markdown("**Risk-return tradeoff.** Each point is a random portfolio.")
        n = 3000
        rng = np.random.default_rng(42)
        rand_w = rng.dirichlet(np.ones(len(mu_s)), n)
        rets = rand_w @ mu_s.values
        vols = np.sqrt(np.einsum("ij,jk,ik->i", rand_w, cov.values, rand_w))
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=vols, y=rets, mode="markers",
                                 marker=dict(size=4, color=rets/vols, colorscale="Viridis")))
        fig.add_trace(go.Scatter(x=[v_s, v_m], y=[r_s, r_m], mode="markers+text",
                                 marker=dict(size=18, color=["gold", "silver"]),
                                 text=["Max Sharpe", "Min Vol"], textposition="top center"))
        fig.update_layout(xaxis_title="Risk", yaxis_title="Expected return", height=520)
        st.plotly_chart(fig, width="stretch")

    with sub[2]:
        c1, c2 = st.columns(2)
        with c1:
            st.markdown("**Max Sharpe**")
            df = pd.DataFrame({"weight": w_sharpe})
            df.index = [drug_names.get(i, i) for i in df.index]
            st.dataframe(df.style.format("{:.3f}"), width="stretch")
            st.metric("Return", f"{r_s:.3f}")
            st.metric("Volatility", f"{v_s:.3f}")
            st.metric("Sharpe", f"{s_s:.3f}")
        with c2:
            st.markdown("**Min Volatility**")
            df = pd.DataFrame({"weight": w_minvol})
            df.index = [drug_names.get(i, i) for i in df.index]
            st.dataframe(df.style.format("{:.3f}"), width="stretch")
            st.metric("Return", f"{r_m:.3f}")
            st.metric("Volatility", f"{v_m:.3f}")

    with sub[3]:
        if len(tickers) >= 2:
            t1 = st.selectbox("Drug A", tickers, index=0, format_func=lambda x: drug_names.get(x, x))
            t2 = st.selectbox("Drug B", tickers, index=1, format_func=lambda x: drug_names.get(x, x))
            rho = st.slider("Correlation", 0.0, 1.0, float(corr.loc[t1, t2]), 0.01)
            corr2 = corr.copy(); corr2.loc[t1, t2] = rho; corr2.loc[t2, t1] = rho
            cov2 = pd.DataFrame(np.outer(sigma_s, sigma_s) * corr2.loc[est.index, est.index].values,
                                index=est.index, columns=est.index)
            ef = EfficientFrontier(mu_s, cov2, weight_bounds=(0, 1))
            try:
                ef.max_sharpe(risk_free_rate=rfr_value)
                w = pd.Series(ef.clean_weights())
                w.index = [drug_names.get(i, i) for i in w.index]
                st.bar_chart(w)
            except Exception as e:
                st.error(f"Solver error: {e}")


# ---------- TAB 3: STRATEGIC ----------
with tabs[2]:
    sub = st.tabs(["⏳ Real Options", "⚖️ FTO", "🧪 Trials", "💰 Pricing"])

    with sub[0]:
        st.markdown("**When should you kill this project?** Each candidate is modeled as a series of phase gates.")
        c1_, c2_ = st.columns([1, 2])
        with c1_:
            peak = st.number_input("Peak sales ($M)", 50, 2000, 200, step=50) * 1e6
            vol  = st.slider("Volatility", 0.20, 0.80, 0.40, 0.05)
        _ph = [
            options.Phase("Phase I", 2.0, float(assumptions.get_value("cost.phase_1_usd", 10e6)), float(assumptions.get_value("pos.phase_1", 0.63))),
            options.Phase("Phase II", 2.0, float(assumptions.get_value("cost.phase_2_usd", 25e6)), float(assumptions.get_value("pos.phase_2", 0.31))),
            options.Phase("Phase III", 3.0, float(assumptions.get_value("cost.phase_3_usd", 100e6)), float(assumptions.get_value("pos.phase_3", 0.58))),
            options.Phase("NDA", 1.0, 5e6, float(assumptions.get_value("pos.nda_bla", 0.85))),
        ]
        r = options.RealOptionsEngine(phases=_ph, peak_sales=peak, volatility=vol,
                                      risk_free_rate=rfr_value, discount_rate=0.10).value()
        mc1, mc2, mc3 = st.columns(3)
        mc1.metric("DCF", f"${r.asset_value:.1f}M")
        mc2.metric("Option value", f"${r.option_value:.1f}M")
        mc3.metric("Flexibility premium", f"${r.abandonment_value:.1f}M")
        if r.option_value > 0:
            st.success(f"✅ PROCEED — option value ${r.option_value:.1f}M")
        else:
            st.error(f"❌ DO NOT START — DCF ${r.asset_value:.1f}M")
        gate_df = pd.DataFrame(r.decision_tree)
        st.dataframe(gate_df, width="stretch", hide_index=True)

    with sub[1]:
        st.markdown("**Commercial IP position.** Screening only — not legal advice.")
        fto_tab = fto.assess_portfolio(candidates, verbose=False)
        mc1, mc2, mc3 = st.columns(3)
        counts = fto_tab["fto_risk"].value_counts().to_dict()
        mc1.metric("🟢 Low", counts.get("low", 0))
        mc2.metric("🟡 Medium", counts.get("medium", 0))
        mc3.metric("🔴 High", counts.get("high", 0))
        disp = fto_tab[["drug_name", "status", "fto_risk", "fto_score", "rationale"]].copy()
        disp.columns = ["Drug", "Stage", "Risk", "Score", "Rationale"]
        st.dataframe(disp, width="stretch", hide_index=True)

    with sub[2]:
        st.markdown("**Trial architecture.** Platform trials share control arms — usually cheaper and faster.")
        n_test = st.slider("Candidates to test", 2, 10, min(len(est), 6))
        designs = trials.recommend_designs(n_candidates=n_test)
        df_t = trials.format_recommendation(designs)
        cheapest = min(designs, key=lambda d: d.expected_cost_usd)
        baseline = max(designs, key=lambda d: d.expected_cost_usd)
        savings = baseline.expected_cost_usd - cheapest.expected_cost_usd
        mc1, mc2, mc3 = st.columns(3)
        mc1.metric("Cheapest", cheapest.name.split("(")[0].strip(), f"${cheapest.expected_cost_usd/1e6:.1f}M")
        mc2.metric("Fastest", min(designs, key=lambda d: d.expected_duration_months).name.split("(")[0].strip())
        mc3.metric("Savings vs fixed", f"${savings/1e6:.1f}M")
        st.dataframe(df_t[["Design", "Arms", "Sample size", "Duration (mo)", "Cost ($M)", "Savings %"]],
                     width="stretch", hide_index=True)

    with sub[3]:
        st.markdown("**Sustainable price per patient per year.** Recovers development cost + WACC.")
        pr_df = pricing.analyze_portfolio(candidates, disease, verbose=False)
        counts = pr_df["viability"].value_counts().to_dict()
        mc1, mc2, mc3 = st.columns(3)
        mc1.metric("🟢 Viable", counts.get("commercially_viable", 0))
        mc2.metric("🟡 Marginal", counts.get("marginal", 0))
        mc3.metric("🔴 Not viable", counts.get("not_viable", 0))
        disp = pr_df[["drug_name", "phase", "sustainable_usd", "reference_usd", "viability"]].copy()
        disp.columns = ["Drug", "Phase", "Sustainable $/yr", "Reference $/yr", "Viability"]
        st.dataframe(disp, width="stretch", hide_index=True)


# ---------- TAB 4: EVIDENCE ----------
with tabs[3]:
    sub = st.tabs(["🌐 Cross-Disease", "📉 Prior Results", "🔬 Synergy"])

    with sub[0]:
        st.markdown("**Other diseases where each candidate is tested.** More indications = more accumulated safety data.")
        if st.button("Analyze cross-disease", key="cd_btn"):
            with st.spinner("Querying Open Targets..."):
                st.session_state["cd_result"] = crosstarget.find_cross_disease_overlaps(candidates, disease, verbose=False)
        cd_df = st.session_state.get("cd_result")
        if cd_df is not None and not cd_df.empty:
            n_over = int((cd_df["n_other_diseases"] > 0).sum())
            st.metric("Cross-disease hits", f"{n_over}/{len(cd_df)}")
            disp = cd_df[["drug_name", "n_other_diseases", "top_other_disease", "all_other_diseases"]].copy()
            disp.columns = ["Drug", "# Others", "Top", "All overlaps"]
            st.dataframe(disp, width="stretch", hide_index=True)
        else:
            st.info("Click above to run cross-disease analysis.")

    with sub[1]:
        st.markdown("**Prior terminated trials.** Weighted by failure reason (futility/safety = strong; enrollment/business = weak).")
        if st.button("Analyze prior results", key="nd_btn"):
            with st.spinner("Querying ClinicalTrials.gov..."):
                st.session_state["nd_result"] = negativedata.analyze_portfolio(candidates, disease, verbose=False)
        nd_df = st.session_state.get("nd_result")
        if nd_df is not None and not nd_df.empty:
            counts = nd_df["risk_flag"].value_counts().to_dict()
            mc1, mc2, mc3, mc4 = st.columns(4)
            mc1.metric("🔴 High", counts.get("high", 0))
            mc2.metric("🟡 Medium", counts.get("medium", 0))
            mc3.metric("🟢 Low", counts.get("low", 0))
            mc4.metric("✅ Clean", counts.get("clean", 0))
            disp = nd_df[["drug_name", "n_failed", "n_relevant", "risk_flag", "weighted_risk", "breakdown"]].copy()
            disp.columns = ["Drug", "# Failed", "# Relevant", "Risk", "Weighted", "Breakdown"]
            st.dataframe(disp, width="stretch", hide_index=True)
        else:
            st.info("Click above to check prior failures.")

    with sub[2]:
        st.markdown("**Pairs to test together.** Peak synergy at correlation 0.1–0.5 — shared context, different mechanism.")
        if st.button("Analyze synergy", key="syn_btn"):
            with st.spinner("Scoring pairs..."):
                st.session_state["syn_result"] = synergy.find_synergy_pairs(candidates, corr, top_n=15, check_trials=False, verbose=False)
        syn_df = st.session_state.get("syn_result")
        if syn_df is not None and not syn_df.empty:
            top = synergy.top_combination_recommendation(syn_df)
            if top.get("found"):
                st.success(f"🎯 Top pair: **{top['drug_a']} + {top['drug_b']}** (score {top['score']:.2f})")
            disp = syn_df[["drug_a", "drug_b", "correlation", "synergy_score", "rationale"]].copy()
            disp.columns = ["Drug A", "Drug B", "Correlation", "Synergy", "Rationale"]
            st.dataframe(disp, width="stretch", hide_index=True)
        else:
            st.info("Click above to score combinations.")


# ---------- TAB 5: ASSUMPTIONS ----------
with tabs[4]:
    st.subheader("Assumption registry")
    st.caption("Every numeric input with source, confidence, and timestamp.")
    reg = assumptions.load_registry()
    rows = []
    def _flat(node, prefix=""):
        for k, v in node.items():
            p = f"{prefix}.{k}" if prefix else k
            if isinstance(v, dict) and "value" in v:
                rows.append({"Key": p, "Value": v["value"], "Confidence": v.get("confidence", "?"),
                             "Source": v.get("source", "")})
            elif isinstance(v, dict):
                _flat(v, p)
    _flat(reg)
    st.dataframe(pd.DataFrame(rows), width="stretch", hide_index=True)


# ---------- TAB 6: REPORT ----------
with tabs[5]:
    st.subheader("📄 Download report")
    st.markdown("Generate a 6-page PDF with methodology, charts, and full provenance.")
    if st.button("📥 Generate PDF report", type="primary", key="pdf_btn"):
        with st.spinner("Generating..."):
            import tempfile
            with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmp:
                tmp_path = Path(tmp.name)
            findings = f"Analysis of {disease} with {len(est)} candidates. Portfolio Sharpe: {s_s:.2f}."
            reporting.build_report(
                path=tmp_path, title=f"RepurposeAlpha — {disease.title()}",
                user=user["username"], disease=disease,
                corr=corr.rename(index=drug_names, columns=drug_names),
                mu=mu_s, sigma=sigma_s, cov=cov,
                w_sharpe=w_sharpe.rename(index=drug_names),
                w_minvol=w_minvol.rename(index=drug_names),
                registry=assumptions.load_registry(), findings=findings,
            )
            st.session_state["pdf_bytes"] = tmp_path.read_bytes()
            st.session_state["pdf_name"] = f"RepurposeAlpha_{disease.replace(' ', '_')}.pdf"
            tmp_path.unlink()
            st.success(f"✅ Ready ({len(st.session_state['pdf_bytes']):,} bytes)")
    if "pdf_bytes" in st.session_state:
        st.download_button("⬇️  Download PDF", data=st.session_state["pdf_bytes"],
                           file_name=st.session_state.get("pdf_name", "report.pdf"),
                           mime="application/pdf", type="primary")


# ---------- TAB 7: ADMIN ----------
if auth.has_role(user, "admin") and len(tabs) == 7:
    with tabs[6]:
        st.subheader("🛡️ Admin panel")
        st.markdown("**Users**")
        st.dataframe(pd.DataFrame(auth.list_users(), columns=["username", "role", "created_at"]),
                     width="stretch", hide_index=True)
        st.markdown("**Audit log**")
        st.dataframe(pd.DataFrame(auth.list_audit(50),
                                  columns=["timestamp", "user", "action", "detail"]),
                     width="stretch", hide_index=True)


st.divider()
st.caption(f"RepurposeAlpha v0.3.0 · {disease} · {len(est)} candidates")

Overwriting C:\Users\Administrator\RepurposeAlpha\app\app.py


In [5]:
from pathlib import Path

# Patch the executive summary to use disease-aware peak sales
exec_file = Path(r"C:\Users\Administrator\RepurposeAlpha\src\executive\__init__.py")
content = exec_file.read_text(encoding="utf-8")

# Add disease-specific peak sales lookup
helper = '''

def estimate_peak_sales(disease_name: str) -> float:
    """Rough peak-sales estimate based on disease category."""
    if not disease_name:
        return 200e6
    d = disease_name.lower()

    # Rare diseases → high per-patient price, small population
    if any(k in d for k in ["syndrome", "dystrophy", "deficiency", "gaucher", "fabry"]):
        return 500e6   # ~$500M peak from orphan pricing

    # Neglected tropical → mass market, low per-patient
    if any(k in d for k in ["malaria", "tuberculosis", "leishmaniasis", "schistosomiasis",
                             "trypanosomiasis", "chagas", "dengue", "ebola"]):
        return 300e6   # ~$300M peak from volume pricing

    # Oncology → premium pricing
    if any(k in d for k in ["cancer", "carcinoma", "leukemia", "lymphoma", "melanoma", "tumor"]):
        return 800e6

    # Common chronic → large volume, moderate price
    if any(k in d for k in ["diabetes", "hypertension", "cardiovascular", "coronary", "asthma"]):
        return 1000e6

    # Default
    return 200e6


def estimate_volatility(disease_name: str) -> float:
    """Volatility varies by disease area."""
    if not disease_name:
        return 0.40
    d = disease_name.lower()
    if any(k in d for k in ["cancer", "oncology", "leukemia", "lymphoma"]):
        return 0.55   # High failure rate → high volatility
    if any(k in d for k in ["malaria", "tuberculosis", "infectious"]):
        return 0.35   # More predictable in infectious disease
    if any(k in d for k in ["syndrome", "dystrophy", "rare"]):
        return 0.45
    return 0.40
'''

if "estimate_peak_sales" not in content:
    content += helper
    exec_file.write_text(content, encoding="utf-8")
    print("✅ Added disease-aware helpers to executive/__init__.py")
else:
    print("ℹ️  Helpers already present")

✅ Added disease-aware helpers to executive/__init__.py


In [6]:
from pathlib import Path

app_file = Path(r"C:\Users\Administrator\RepurposeAlpha\app\app.py")
content = app_file.read_text(encoding="utf-8")

# Replace the hardcoded 200e6 with disease-aware
old = 'ro_result = options.RealOptionsEngine(phases=_phases_ro, peak_sales=200e6, volatility=0.40,\n                                                  risk_free_rate=rfr_value, discount_rate=0.10).value()'
new = '''_peak = executive.estimate_peak_sales(disease)
            _vol = executive.estimate_volatility(disease)
            ro_result = options.RealOptionsEngine(phases=_phases_ro, peak_sales=_peak, volatility=_vol,
                                                  risk_free_rate=rfr_value, discount_rate=0.10).value()'''

if old in content:
    content = content.replace(old, new)
    app_file.write_text(content, encoding="utf-8")
    print("✅ Patched app to use disease-aware peak sales")
elif "_peak = executive.estimate_peak_sales" in content:
    print("ℹ️  Already patched")
else:
    print("⚠️  Could not find target line. Search manually.")

✅ Patched app to use disease-aware peak sales


In [7]:
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\Administrator\RepurposeAlpha")
exec_dir = PROJECT_ROOT / "src" / "executive"
exec_dir.mkdir(parents=True, exist_ok=True)
(exec_dir / "__init__.py").touch()
print(f"✅ Folder: {exec_dir}")

✅ Folder: C:\Users\Administrator\RepurposeAlpha\src\executive


In [8]:
%%writefile C:\Users\Administrator\RepurposeAlpha\src\executive\__init__.py
"""
RepurposeAlpha — Executive Summary engine v2.

Runs all seven features internally, then produces:
- Verdict (RECOMMENDED / CAUTION / NOT RECOMMENDED)
- Plain-English narrative
- Feature scorecard (one line per feature)
- Top candidates with strengths + risks
- Warnings (prioritized)
- Next steps (numbered)

Designed to be the single view a decision-maker reads first.
"""
import sys
import pandas as pd
import numpy as np
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Optional

# Local imports — features must be on sys.path
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
import assumptions
import options
import fto
import trials
import crosstarget
import negativedata
import synergy
import pricing


@dataclass
class FeatureCard:
    icon: str
    title: str
    status: str       # "positive" | "caution" | "negative" | "neutral"
    headline: str     # what the feature found, punchy
    detail: str       # why it matters, plain English


@dataclass
class CandidateVerdict:
    name: str
    phase: str
    weight: float
    expected_return: float
    strengths: List[str] = field(default_factory=list)
    risks: List[str] = field(default_factory=list)


@dataclass
class Warning:
    level: str
    icon: str
    text: str


@dataclass
class ExecutiveSummary:
    disease: str
    n_candidates: int
    verdict: str
    verdict_color: str
    one_liner: str
    narrative: str
    feature_cards: List[FeatureCard]
    top_candidates: List[CandidateVerdict]
    warnings: List[Warning]
    next_steps: List[str]
    confidence: str


def _safe(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception:
        return None


def estimate_peak_sales(disease_name):
    """Rough peak-sales estimate based on disease category."""
    if not disease_name:
        return 200e6
    d = disease_name.lower()
    if any(k in d for k in ["syndrome", "dystrophy", "deficiency", "gaucher", "fabry", "pompe"]):
        return 500e6
    if any(k in d for k in ["malaria", "tuberculosis", "leishmaniasis", "schistosomiasis",
                             "trypanosomiasis", "chagas", "dengue", "ebola"]):
        return 300e6
    if any(k in d for k in ["cancer", "carcinoma", "leukemia", "lymphoma", "melanoma", "tumor"]):
        return 800e6
    if any(k in d for k in ["diabetes", "hypertension", "cardiovascular", "coronary", "asthma"]):
        return 1000e6
    return 200e6


def estimate_volatility(disease_name):
    if not disease_name:
        return 0.40
    d = disease_name.lower()
    if any(k in d for k in ["cancer", "oncology", "leukemia", "lymphoma"]):
        return 0.55
    if any(k in d for k in ["malaria", "tuberculosis", "infectious"]):
        return 0.35
    if any(k in d for k in ["syndrome", "dystrophy", "rare"]):
        return 0.45
    return 0.40


def build_summary(disease, candidates, corr, est, w_sharpe,
                   r_s, v_s, s_s, rfr_value):
    """
    Run all seven features and build the full executive summary.
    """
    drug_names = dict(zip(candidates["chembl_id"], candidates["drug_name"]))
    n = len(candidates)

    # ---------- Run every feature ----------
    fto_df = _safe(fto.assess_portfolio, candidates, verbose=False)

    priors_df = _safe(negativedata.analyze_portfolio, candidates, disease, verbose=False)

    synergy_df = _safe(synergy.find_synergy_pairs, candidates, corr,
                        top_n=3, check_trials=False, verbose=False)

    pricing_df = _safe(pricing.analyze_portfolio, candidates, disease, verbose=False)

    # Real options
    ro_result = None
    try:
        _ph = [
            options.Phase("Phase I", 2.0, float(assumptions.get_value("cost.phase_1_usd", 10e6)),
                          float(assumptions.get_value("pos.phase_1", 0.63))),
            options.Phase("Phase II", 2.0, float(assumptions.get_value("cost.phase_2_usd", 25e6)),
                          float(assumptions.get_value("pos.phase_2", 0.31))),
            options.Phase("Phase III", 3.0, float(assumptions.get_value("cost.phase_3_usd", 100e6)),
                          float(assumptions.get_value("pos.phase_3", 0.58))),
            options.Phase("NDA", 1.0, 5e6, float(assumptions.get_value("pos.nda_bla", 0.85))),
        ]
        ro_result = options.RealOptionsEngine(
            phases=_ph,
            peak_sales=estimate_peak_sales(disease),
            volatility=estimate_volatility(disease),
            risk_free_rate=rfr_value,
            discount_rate=0.10,
        ).value()
    except Exception:
        ro_result = None

    # Trial designs
    trial_designs = _safe(trials.recommend_designs, n_candidates=min(len(est), 6))

    # ---------- Feature scorecard ----------
    cards = []

    # 1. Portfolio
    top_w = w_sharpe.sort_values(ascending=False)
    top_names = [drug_names.get(i, i) for i in top_w.index[:3] if top_w[i] > 0.05]
    if s_s > 1.5:
        p_status, p_headline = "positive", f"Sharpe {s_s:.2f} — strong risk-adjusted return"
    elif s_s > 0.8:
        p_status, p_headline = "positive", f"Sharpe {s_s:.2f} — defensible portfolio"
    elif s_s > 0.4:
        p_status, p_headline = "caution", f"Sharpe {s_s:.2f} — modest signal"
    else:
        p_status, p_headline = "negative", f"Sharpe {s_s:.2f} — weak signal"
    cards.append(FeatureCard(
        "📈", "Portfolio Optimization", p_status, p_headline,
        f"Top picks: {', '.join(top_names) if top_names else 'none above threshold'}",
    ))

    # 2. Real Options
    if ro_result is None:
        cards.append(FeatureCard("⏳", "Real Options", "neutral", "Not available", "—"))
    elif ro_result.option_value > 0:
        cards.append(FeatureCard(
            "⏳", "Real Options", "positive",
            f"PROCEED — option value ${ro_result.option_value:.0f}M",
            f"Flexibility over DCF adds ${ro_result.abandonment_value:.0f}M",
        ))
    else:
        cards.append(FeatureCard(
            "⏳", "Real Options", "negative",
            f"DO NOT START — DCF ${ro_result.asset_value:.0f}M",
            "Even with the option to abandon, this doesn't clear cost of capital.",
        ))

    # 3. FTO
    if fto_df is None or fto_df.empty:
        cards.append(FeatureCard("⚖️", "Freedom-to-Operate", "neutral", "Not available", "—"))
    else:
        n_low = int((fto_df["fto_risk"] == "low").sum())
        n_high = int((fto_df["fto_risk"] == "high").sum())
        if n_high == 0:
            cards.append(FeatureCard(
                "⚖️", "Freedom-to-Operate", "positive",
                f"{n_low}/{n} with strong IP position",
                "No generic-competition blockers.",
            ))
        elif n_high >= n // 2:
            cards.append(FeatureCard(
                "⚖️", "Freedom-to-Operate", "negative",
                f"{n_high}/{n} are generic",
                "Limited commercial protection — no compound patent.",
            ))
        else:
            cards.append(FeatureCard(
                "⚖️", "Freedom-to-Operate", "caution",
                f"{n_low} protected · {n_high} generic",
                "Mixed IP position — review before investment.",
            ))

    # 4. Trials
    if trial_designs:
        cheapest = min(trial_designs, key=lambda d: d.expected_cost_usd)
        baseline = max(trial_designs, key=lambda d: d.expected_cost_usd)
        savings = baseline.expected_cost_usd - cheapest.expected_cost_usd
        cards.append(FeatureCard(
            "🧪", "Trial Design", "positive",
            f"Platform trial saves ${savings/1e6:.1f}M",
            f"vs. {n if n <= 6 else 6} independent RCTs",
        ))
    else:
        cards.append(FeatureCard("🧪", "Trial Design", "neutral", "Not available", "—"))

    # 5. Cross-Disease
    cd_df = _safe(crosstarget.find_cross_disease_overlaps, candidates, disease, verbose=False)
    if cd_df is None or cd_df.empty:
        cards.append(FeatureCard("🌐", "Cross-Disease", "neutral", "Not analyzed", "Click Evidence tab"))
    else:
        n_over = int((cd_df["n_other_diseases"] > 0).sum())
        if n_over >= n * 0.7:
            cards.append(FeatureCard(
                "🌐", "Cross-Disease", "positive",
                f"{n_over}/{n} have other indications",
                "Accumulated safety data — lower regulatory risk.",
            ))
        else:
            cards.append(FeatureCard(
                "🌐", "Cross-Disease", "neutral",
                f"{n_over}/{n} have other indications",
                "Limited prior evidence in other diseases.",
            ))

    # 6. Prior Results
    if priors_df is None or priors_df.empty:
        cards.append(FeatureCard("📉", "Prior Results", "neutral", "Not analyzed", "Click Evidence tab"))
    else:
        n_high = int((priors_df["risk_flag"] == "high").sum())
        n_clean = int((priors_df["risk_flag"] == "clean").sum())
        if n_high == 0:
            cards.append(FeatureCard(
                "📉", "Prior Results", "positive",
                f"{n_clean}/{n} clean — no strong failures",
                "No futility or safety signals in prior trials.",
            ))
        elif n_high >= 2:
            cards.append(FeatureCard(
                "📉", "Prior Results", "negative",
                f"{n_high} with strong prior failures",
                "Futility or safety issues — review before proceeding.",
            ))
        else:
            cards.append(FeatureCard(
                "📉", "Prior Results", "caution",
                f"{n_high} with prior failure signal",
                "At least one candidate has meaningful prior failure history.",
            ))

    # 7. Synergy
    if synergy_df is None or synergy_df.empty:
        cards.append(FeatureCard("🔬", "Synergy", "neutral", "Not analyzed", "Click Evidence tab"))
    else:
        top = synergy_df.iloc[0]
        score = float(top["synergy_score"])
        if score >= 0.55:
            cards.append(FeatureCard(
                "🔬", "Synergy", "positive",
                f"Top pair: {top['drug_a'][:20]} + {top['drug_b'][:20]}",
                f"Synergy score {score:.2f} — good combination candidate.",
            ))
        else:
            cards.append(FeatureCard(
                "🔬", "Synergy", "neutral",
                f"Top pair scored {score:.2f}",
                "No strong synergy signal in this candidate set.",
            ))

    # 8. Pricing
    if pricing_df is None or pricing_df.empty:
        cards.append(FeatureCard("💰", "Pricing", "neutral", "Not available", "—"))
    else:
        n_viable = int((pricing_df["viability"] == "commercially_viable").sum())
        n_not = int((pricing_df["viability"] == "not_viable").sum())
        if n_not == 0:
            cards.append(FeatureCard(
                "💰", "Pricing", "positive",
                f"{n_viable}/{n} commercially viable",
                "Sustainable prices are within market reference range.",
            ))
        elif n_viable >= n // 2:
            cards.append(FeatureCard(
                "💰", "Pricing", "caution",
                f"{n_viable} viable · {n_not} not viable",
                "Mixed commercial position at reference prices.",
            ))
        else:
            cards.append(FeatureCard(
                "💰", "Pricing", "negative",
                f"{n_not}/{n} not commercially viable",
                "Price needed exceeds market reference range.",
            ))

    # ---------- Warnings ----------
    warnings = []
    for card in cards:
        if card.status == "negative":
            warnings.append(Warning("high", card.icon, f"{card.title}: {card.headline}"))
        elif card.status == "caution":
            warnings.append(Warning("medium", card.icon, f"{card.title}: {card.headline}"))

    # ---------- Overall verdict ----------
    n_neg = sum(1 for c in cards if c.status == "negative")
    n_pos = sum(1 for c in cards if c.status == "positive")
    n_cau = sum(1 for c in cards if c.status == "caution")

    if n_neg >= 2:
        verdict, color = "NOT RECOMMENDED", "red"
    elif n_neg == 1 or n_cau >= 3:
        verdict, color = "CAUTION", "yellow"
    elif n_pos >= 4:
        verdict, color = "RECOMMENDED", "green"
    else:
        verdict, color = "REVIEW", "yellow"

    # ---------- Narrative ----------
    narrative_parts = []
    narrative_parts.append(
        f"Analysis of **{disease.title()}** identified **{n} candidate drugs** for repurposing."
    )

    # Portfolio finding
    if top_names:
        narrative_parts.append(
            f"Portfolio optimization ranks **{top_names[0]}** first "
            f"(weight {top_w[top_w.index[0]]:.2f}), with a portfolio Sharpe ratio of **{s_s:.2f}**."
        )

    # Real options finding
    if ro_result is not None:
        if ro_result.option_value > 0:
            narrative_parts.append(
                f"Real options analysis says **PROCEED** — the flexibility to abandon "
                f"is worth ${ro_result.abandonment_value:.0f}M over a naive DCF."
            )
        else:
            narrative_parts.append(
                f"Real options analysis says **DO NOT START** from scratch — the DCF is "
                f"${ro_result.asset_value:.0f}M. The option value is zero."
            )

    # FTO finding
    if fto_df is not None and not fto_df.empty:
        n_high = int((fto_df["fto_risk"] == "high").sum())
        if n_high > 0:
            narrative_parts.append(
                f"On commercial protection, **{n_high} of {n} candidates are generic** — no compound patent."
            )
        else:
            narrative_parts.append(
                f"All candidates have **at least partial IP protection** — a good sign for commercial viability."
            )

    # Prior results
    if priors_df is not None and not priors_df.empty:
        n_high_p = int((priors_df["risk_flag"] == "high").sum())
        if n_high_p > 0:
            narrative_parts.append(
                f"Prior trial history shows **{n_high_p} candidate(s) with meaningful failure signals**."
            )

    # Trials
    if trial_designs:
        cheapest = min(trial_designs, key=lambda d: d.expected_cost_usd)
        baseline = max(trial_designs, key=lambda d: d.expected_cost_usd)
        savings = baseline.expected_cost_usd - cheapest.expected_cost_usd
        narrative_parts.append(
            f"If you proceed to validation, using a **{cheapest.name.split('(')[0].strip().lower()}** "
            f"would save **${savings/1e6:.1f}M** vs. independent trials."
        )

    narrative_parts.append(f"**Overall verdict: {verdict}.**")
    narrative = " ".join(narrative_parts)

    # ---------- One-liner ----------
    if verdict == "RECOMMENDED":
        one_liner = "The portfolio is well-positioned. Proceed with validation planning."
    elif verdict == "CAUTION":
        one_liner = "Proceed with caution — resolve the flagged issues first."
    elif verdict == "NOT RECOMMENDED":
        one_liner = "Multiple critical issues. Reassess before committing resources."
    else:
        one_liner = "Review the feature scorecard for details."

    # ---------- Top candidates ----------
    top_ids = [i for i in top_w.index if top_w[i] > 0.02][:5]
    top_candidates = []
    for cid in top_ids:
        name = drug_names.get(cid, cid)
        weight = float(top_w[cid])
        phase_key = est.loc[cid, "phase_key"] if cid in est.index else "unknown"
        phase = phase_key.replace("_", " ").title()
        mu = float(est.loc[cid, "mu"]) if cid in est.index else 0.0

        strengths = []
        risks = []

        if weight > 0.15:
            strengths.append(f"Highest portfolio priority (weight {weight:.2f})")

        # FTO
        if fto_df is not None and not fto_df.empty:
            row = fto_df[fto_df["chembl_id"] == cid]
            if not row.empty:
                score = float(row.iloc[0]["fto_score"])
                if score >= 0.65:
                    strengths.append("Strong IP position")
                elif score < 0.30:
                    risks.append("Generic — no commercial protection")

        # Correlation
        correlated = [(drug_names.get(j, j), float(corr.loc[cid, j]))
                      for j in corr.columns if j != cid and corr.loc[cid, j] > 0.4]
        if correlated:
            peer, r = max(correlated, key=lambda x: x[1])
            risks.append(f"Correlated with {peer} (r={r:.2f}) — potentially redundant")

        # Prior results
        if priors_df is not None and not priors_df.empty:
            row = priors_df[priors_df["chembl_id"] == cid]
            if not row.empty and row.iloc[0]["risk_flag"] in ("high", "medium"):
                risks.append(f"Prior failure signal ({row.iloc[0]['risk_flag']})")

        top_candidates.append(CandidateVerdict(
            name=name, phase=phase, weight=weight,
            expected_return=mu, strengths=strengths, risks=risks,
        ))

    # ---------- Next steps ----------
    next_steps = []
    if top_names:
        next_steps.append(
            f"Prioritize validation budget on **{', '.join(top_names[:3])}**"
        )
    if priors_df is not None and int((priors_df["risk_flag"] == "high").sum()) > 0:
        next_steps.append(
            "Review prior failure details for flagged candidates before committing"
        )
    if fto_df is not None and int((fto_df["fto_risk"] == "high").sum()) > 0:
        next_steps.append(
            "Define IP strategy for generic candidates (formulation, method-of-use)"
        )
    if trial_designs:
        cheapest = min(trial_designs, key=lambda d: d.expected_cost_usd)
        next_steps.append(
            f"Plan a **{cheapest.name.split('(')[0].strip().lower()}** — reduces cost by "
            f"${(max(d.expected_cost_usd for d in trial_designs) - cheapest.expected_cost_usd)/1e6:.1f}M"
        )
    if synergy_df is not None and not synergy_df.empty:
        top_pair = synergy_df.iloc[0]
        next_steps.append(
            f"Consider combination testing: **{top_pair['drug_a']} + {top_pair['drug_b']}**"
        )
    if not next_steps:
        next_steps.append("Review each tab for detailed analysis")

    # ---------- Confidence ----------
    available = sum(1 for c in cards if c.status != "neutral")
    if available >= 7:
        confidence = "high"
    elif available >= 5:
        confidence = "medium"
    else:
        confidence = "low"

    return ExecutiveSummary(
        disease=disease,
        n_candidates=n,
        verdict=verdict,
        verdict_color=color,
        one_liner=one_liner,
        narrative=narrative,
        feature_cards=cards,
        top_candidates=top_candidates,
        warnings=warnings,
        next_steps=next_steps,
        confidence=confidence,
    )

Overwriting C:\Users\Administrator\RepurposeAlpha\src\executive\__init__.py


In [9]:
from pathlib import Path
import re

app_file = Path(r"C:\Users\Administrator\RepurposeAlpha\app\app.py")
content = app_file.read_text(encoding="utf-8")

# Find the executive tab section and replace it
start_marker = "# ---------- TAB 1: EXECUTIVE SUMMARY ----------"
end_marker = "# ---------- TAB 2: PORTFOLIO ----------"

if start_marker not in content or end_marker not in content:
    print("⚠️  Could not find markers. Manual patch needed.")
else:
    start_idx = content.find(start_marker)
    end_idx = content.find(end_marker)

    new_block = '''# ---------- TAB 1: EXECUTIVE SUMMARY ----------
with tabs[0]:
    with st.spinner("Running all seven features..."):
        summary = executive.build_summary(
            disease=disease, candidates=candidates, corr=corr,
            est=est, w_sharpe=w_sharpe,
            r_s=r_s, v_s=v_s, s_s=s_s, rfr_value=rfr_value,
        )

    # ---- VERDICT BANNER ----
    color_map = {"green": "#10b981", "yellow": "#f59e0b", "red": "#ef4444"}
    vc = color_map.get(summary.verdict_color, "#64748b")
    st.markdown(
        f"""<div style="background:{vc}22; border-left:5px solid {vc};
        padding:22px 26px; border-radius:8px; margin-bottom:24px;">
        <div style="font-size:11px; color:#94a3b8; letter-spacing:1.8px; text-transform:uppercase;">Overall verdict</div>
        <div style="font-size:34px; font-weight:700; color:{vc}; margin:6px 0 10px 0;">{summary.verdict}</div>
        <div style="font-size:15px; color:#cbd5e1;">{summary.one_liner}</div>
        </div>""", unsafe_allow_html=True)

    # ---- NARRATIVE ----
    st.markdown("### 📖 Summary")
    st.markdown(summary.narrative)

    # ---- FEATURE SCORECARD ----
    st.markdown("### 🎯 Feature Scorecard")
    st.caption("What each of the seven features found, in one line.")
    status_colors = {
        "positive": ("#10b981", "✅"),
        "caution":  ("#f59e0b", "⚠️"),
        "negative": ("#ef4444", "❌"),
        "neutral":  ("#64748b", "○"),
    }
    cols = st.columns(2)
    for i, card in enumerate(summary.feature_cards):
        with cols[i % 2]:
            color, badge = status_colors[card.status]
            st.markdown(
                f"""<div style="background:#1e293b; border-left:4px solid {color};
                padding:14px 18px; border-radius:6px; margin-bottom:12px;">
                <div style="display:flex; align-items:center; gap:10px; margin-bottom:6px;">
                    <span style="font-size:18px;">{card.icon}</span>
                    <span style="font-weight:600; color:#f1f5f9;">{card.title}</span>
                    <span style="margin-left:auto; color:{color};">{badge}</span>
                </div>
                <div style="color:{color}; font-size:14px; font-weight:500; margin-bottom:4px;">{card.headline}</div>
                <div style="color:#94a3b8; font-size:13px;">{card.detail}</div>
                </div>""", unsafe_allow_html=True)

    st.divider()

    # ---- TOP CANDIDATES ----
    st.markdown("### 🏆 Top candidates")
    if summary.top_candidates:
        for i, c in enumerate(summary.top_candidates[:5], 1):
            with st.expander(f"**{i}. {c.name}** · {c.phase} · weight {c.weight:.3f}", expanded=(i <= 2)):
                c1_, c2_ = st.columns(2)
                with c1_:
                    st.markdown("**Strengths**")
                    if c.strengths:
                        for s in c.strengths:
                            st.markdown(f"✅ {s}")
                    else:
                        st.caption("No specific strengths flagged")
                with c2_:
                    st.markdown("**Risks**")
                    if c.risks:
                        for r in c.risks:
                            st.markdown(f"⚠️ {r}")
                    else:
                        st.caption("No specific risks flagged")
                st.caption(f"Expected annual return: {c.expected_return:.3f}")
    else:
        st.info("No candidates passed the portfolio threshold.")

    st.divider()

    # ---- WARNINGS ----
    if summary.warnings:
        st.markdown("### ⚠️ Items to review")
        for w in summary.warnings:
            if w.level == "high":
                st.warning(f"{w.icon} {w.text}")
            else:
                st.info(f"{w.icon} {w.text}")

    # ---- NEXT STEPS ----
    st.markdown("### 📋 Recommended next steps")
    for i, step in enumerate(summary.next_steps, 1):
        st.markdown(f"**{i}.** {step}")

    st.divider()
    st.caption(f"Confidence: **{summary.confidence}** · Full details in the tabs below · PDF in Report tab")


'''

    new_content = content[:start_idx] + new_block + content[end_idx:]
    app_file.write_text(new_content, encoding="utf-8")
    print(f"✅ Executive tab rewritten ({len(new_content):,} bytes)")

✅ Executive tab rewritten (23,398 bytes)


In [10]:
import subprocess
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\Administrator\RepurposeAlpha")

def run(cmd, cwd=PROJECT_ROOT):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout.strip() or r.stderr.strip())

print("--- current status ---")
print(run("git status -sb"))
print("\n--- untracked/modified files ---")
print(run("git status --short"))
print("\n--- last 5 commits ---")
print(run("git log --oneline -5"))

--- current status ---
## main...origin/main
 M app/app.py
 M app/users.db.enc
?? data/cache/e44d967f3e8a_tb/
?? src/executive/

--- untracked/modified files ---
M app/app.py
 M app/users.db.enc
?? data/cache/e44d967f3e8a_tb/
?? src/executive/

--- last 5 commits ---
8568982 docs: technical white paper â€” features, math, assumptions, market comparison
262c16b docs: HTML explainer for all 8 features
79a4cc6 docs: README seven-feature section + KEMRI email draft
0348e14 Feature 7/7: Cost-based pricing framework â€” ALL 7 FEATURES COMPLETE
28eebc8 Feature 6/7: Combination synergy predictor (validated: found Sulfadiazine+Trimethoprim = co-trimoxazole)


In [11]:
from pathlib import Path
import subprocess

PROJECT_ROOT = Path(r"C:\Users\Administrator\RepurposeAlpha")

def run(cmd, cwd=PROJECT_ROOT):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout.strip() or r.stderr.strip())

# 1. Update .gitignore
gitignore = PROJECT_ROOT / ".gitignore"
content = gitignore.read_text(encoding="utf-8")

additions = """
# Auth database — never commit (contains user hashes even if encrypted)
app/users.db
app/users.db.enc
app/*.db
app/*.db.enc

# Analysis cache — regenerable
data/cache/

# Secrets
.env
.secrets/
*.key
*.pem
"""

if "app/users.db.enc" not in content:
    content = content.rstrip() + "\n" + additions
    gitignore.write_text(content, encoding="utf-8")
    print("✅ .gitignore updated")
else:
    print("ℹ️  .gitignore already has the rules")

# 2. Remove users.db.enc from tracking (keeps the file on disk)
print("\n--- removing users.db.enc from git tracking ---")
print(run("git rm --cached app/users.db.enc"))

# 3. Remove any tracked cache files
print("\n--- removing data/cache from tracking (if tracked) ---")
print(run("git rm -r --cached data/cache 2>&1") or "(nothing tracked)")

print("\n--- new status ---")
print(run("git status --short"))

✅ .gitignore updated

--- removing users.db.enc from git tracking ---
rm 'app/users.db.enc'

--- removing data/cache from tracking (if tracked) ---
rm 'data/cache/.gitkeep'
rm 'data/cache/138de2dc4fd4_prader-willi_syndrome/candidates.csv'
rm 'data/cache/138de2dc4fd4_prader-willi_syndrome/correlation.csv'
rm 'data/cache/138de2dc4fd4_prader-willi_syndrome/metadata.json'
rm 'data/cache/182a1e726ac1_schizophrenia/candidates.csv'
rm 'data/cache/182a1e726ac1_schizophrenia/correlation.csv'
rm 'data/cache/182a1e726ac1_schizophrenia/metadata.json'
rm 'data/cache/22810bcbd211_malaria/candidates.csv'
rm 'data/cache/22810bcbd211_malaria/correlation.csv'
rm 'data/cache/22810bcbd211_malaria/metadata.json'
rm 'data/cache/32ab39bcfd75_hiv/candidates.csv'
rm 'data/cache/32ab39bcfd75_hiv/correlation.csv'
rm 'data/cache/32ab39bcfd75_hiv/metadata.json'
rm 'data/cache/3bed3a5d0368_ebola/candidates.csv'
rm 'data/cache/3bed3a5d0368_ebola/correlation.csv'
rm 'data/cache/3bed3a5d0368_ebola/metadata.json'
rm 'd

In [12]:
import subprocess
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\Administrator\RepurposeAlpha")

def run(cmd, cwd=PROJECT_ROOT):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout.strip() or r.stderr.strip())

# Final safety check
print("--- final check: sensitive files NOT tracked ---")
tracked = run("git ls-files").split("\n")
sensitive = [f for f in tracked if "users.db" in f or f.endswith(".env") or f.endswith(".key")]
if sensitive:
    print(f"🚨 STILL TRACKED: {sensitive}")
else:
    print("✅ No sensitive files tracked")

print("\n--- staging ---")
print(run("git add ."))

print("\n--- what's staged ---")
print(run("git diff --cached --name-only"))

print("\n--- committing ---")
print(run('git commit -m "v0.3.1: Executive Summary v2 (narrative + scorecard) + secure users.db"'))

print("\n--- last commit ---")
print(run("git log --oneline -1"))

--- final check: sensitive files NOT tracked ---
✅ No sensitive files tracked

--- staging ---


--- what's staged ---
.gitignore
app/app.py
app/users.db.enc
data/cache/.gitkeep
data/cache/138de2dc4fd4_prader-willi_syndrome/candidates.csv
data/cache/138de2dc4fd4_prader-willi_syndrome/correlation.csv
data/cache/138de2dc4fd4_prader-willi_syndrome/metadata.json
data/cache/182a1e726ac1_schizophrenia/candidates.csv
data/cache/182a1e726ac1_schizophrenia/correlation.csv
data/cache/182a1e726ac1_schizophrenia/metadata.json
data/cache/22810bcbd211_malaria/candidates.csv
data/cache/22810bcbd211_malaria/correlation.csv
data/cache/22810bcbd211_malaria/metadata.json
data/cache/32ab39bcfd75_hiv/candidates.csv
data/cache/32ab39bcfd75_hiv/correlation.csv
data/cache/32ab39bcfd75_hiv/metadata.json
data/cache/3bed3a5d0368_ebola/candidates.csv
data/cache/3bed3a5d0368_ebola/correlation.csv
data/cache/3bed3a5d0368_ebola/metadata.json
data/cache/3faf175ad7c3_schistosomiasis/candidates.csv
data/cache/3faf175ad

In [13]:
import subprocess
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\Administrator\RepurposeAlpha")

def run(cmd, cwd=PROJECT_ROOT):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout.strip() or r.stderr.strip())

print("--- status ---")
print(run("git status -sb"))

print("\n--- sensitive files still on disk? ---")
from pathlib import Path as P
print(f"  users.db.enc exists on disk: {(PROJECT_ROOT/'app'/'users.db.enc').exists()}")
print(f"  users.db.enc tracked by git: {'app/users.db.enc' in run('git ls-files')}")

print("\n--- last 3 commits ---")
print(run("git log --oneline -3"))

--- status ---
## main...origin/main

--- sensitive files still on disk? ---
  users.db.enc exists on disk: True
  users.db.enc tracked by git: False

--- last 3 commits ---
c45c8bb v0.3.1: Executive Summary v2 (narrative + scorecard) + secure users.db
8568982 docs: technical white paper â€” features, math, assumptions, market comparison
262c16b docs: HTML explainer for all 8 features


In [14]:
from pathlib import Path

f = Path(r"C:\Users\Administrator\RepurposeAlpha\src\executive\__init__.py")
content = f.read_text(encoding="utf-8")

# Replace all ** with <strong>/</strong> in narrative parts
# The narrative uses **bold** which breaks in Streamlit when combined with other text

# Add a helper function to convert ** to <strong>
helper = '''

def _md_to_html(text: str) -> str:
    """Convert **bold** markdown to <strong> tags for Streamlit rendering."""
    import re
    # Replace **text** with <strong>text</strong>
    return re.sub(r"\\*\\*(.+?)\\*\\*", r"<strong>\\1</strong>", text)

'''

if "_md_to_html" not in content:
    # Add helper after imports
    marker = "def _safe(fn, *args, **kwargs):"
    content = content.replace(marker, helper + "\n" + marker)

# Change the narrative assignment to use HTML
old = "    narrative = \" \".join(narrative_parts)"
new = "    narrative = _md_to_html(\" \".join(narrative_parts))"

if old in content:
    content = content.replace(old, new)
    print("✅ Narrative now uses HTML bold")

f.write_text(content, encoding="utf-8")
print(f"✅ Updated executive/__init__.py ({len(content):,} bytes)")

✅ Narrative now uses HTML bold
✅ Updated executive/__init__.py (19,048 bytes)


In [15]:
from pathlib import Path

app_file = Path(r"C:\Users\Administrator\RepurposeAlpha\app\app.py")
content = app_file.read_text(encoding="utf-8")

# The Executive tab renders narrative with st.markdown — switch to HTML-safe
old = '''    st.markdown("### 📖 Summary")
    st.markdown(summary.narrative)'''

new = '''    st.markdown("### 📖 Summary")
    st.markdown(summary.narrative, unsafe_allow_html=True)'''

if old in content and "unsafe_allow_html=True" not in content.split("### 📖 Summary")[1][:200]:
    content = content.replace(old, new)
    app_file.write_text(content, encoding="utf-8")
    print("✅ App now renders narrative HTML")
elif "st.markdown(summary.narrative, unsafe_allow_html=True)" in content:
    print("ℹ️  Already patched")
else:
    print("⚠️  Could not find the narrative render line — check manually")

✅ App now renders narrative HTML


In [16]:
import subprocess
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\Administrator\RepurposeAlpha")

def run(cmd, cwd=PROJECT_ROOT):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout.strip() or r.stderr.strip())

print(run("git add ."))
print(run('git commit -m "fix: Executive Summary narrative markdown rendering (HTML bold)"'))
print("\n⚠️  Push from PowerShell.")


[main 1fa0946] fix: Executive Summary narrative markdown rendering (HTML bold)
 2 files changed, 11 insertions(+), 2 deletions(-)

⚠️  Push from PowerShell.


In [17]:
from pathlib import Path

f = Path(r"C:\Users\Administrator\RepurposeAlpha\src\executive\__init__.py")
content = f.read_text(encoding="utf-8")

# Replace the _md_to_html helper with a safer version that also escapes $
old_helper = '''def _md_to_html(text: str) -> str:
    """Convert **bold** markdown to <strong> tags for Streamlit rendering."""
    import re
    # Replace **text** with <strong>text</strong>
    return re.sub(r"\\*\\*(.+?)\\*\\*", r"<strong>\\1</strong>", text)'''

new_helper = '''def _md_to_html(text: str) -> str:
    """Convert **bold** markdown to <strong> tags AND escape $ signs
    so Streamlit's markdown parser doesn't interpret them as math mode."""
    import re
    # First, replace **bold** with <strong>bold</strong>
    text = re.sub(r"\\*\\*(.+?)\\*\\*", r"<strong>\\1</strong>", text)
    # Then escape any remaining $ signs so markdown math-mode doesn't fire
    text = text.replace("$", "\\\\$")
    return text'''

if old_helper in content:
    content = content.replace(old_helper, new_helper)
    print("✅ Updated _md_to_html helper (now escapes $ signs)")
else:
    print("⚠️  Helper not found in expected format. Trying alternative...")
    # Find and replace just the return line
    import re
    content = re.sub(
        r'def _md_to_html\(text: str\) -> str:.*?return re\.sub\(r"\\\\\*\\\\\*\(\.\+\?\)\\\\\*\\\\\*", r"<strong>\\\\1</strong>", text\)',
        new_helper,
        content,
        flags=re.DOTALL,
    )

f.write_text(content, encoding="utf-8")
print(f"✅ File saved ({len(content):,} bytes)")

✅ Updated _md_to_html helper (now escapes $ signs)
✅ File saved (19,248 bytes)


In [18]:
from pathlib import Path

app_file = Path(r"C:\Users\Administrator\RepurposeAlpha\app\app.py")
content = app_file.read_text(encoding="utf-8")

old = '''    st.markdown("### 📖 Summary")
    st.markdown(summary.narrative, unsafe_allow_html=True)'''

new = '''    st.markdown("### 📖 Summary")
    st.markdown(f"<div style='font-size:15px; line-height:1.75; color:#cbd5e1;'>{summary.narrative}</div>", unsafe_allow_html=True)'''

if old in content:
    content = content.replace(old, new)
    app_file.write_text(content, encoding="utf-8")
    print("✅ Narrative now wrapped in HTML div (markdown math disabled)")
elif "font-size:15px; line-height:1.75" in content:
    print("ℹ️  Already wrapped")
else:
    print("⚠️  Target not found. Manual patch needed.")

✅ Narrative now wrapped in HTML div (markdown math disabled)


In [19]:
from pathlib import Path

cache_file = Path(r"C:\Users\Administrator\RepurposeAlpha\src\cache\__init__.py")
content = cache_file.read_text(encoding="utf-8")

# Replace the _key function to include settings
old_key = '''def _key(disease_name: str) -> str:
    slug = disease_name.lower().strip().replace(" ", "_")
    return hashlib.md5(slug.encode()).hexdigest()[:12] + "_" + slug[:40]


def cache_path(disease_name: str) -> Path:
    return CACHE_DIR / f"{_key(disease_name)}"


def has_cache(disease_name: str) -> bool:
    p = cache_path(disease_name)
    return (p / "candidates.csv").exists() and (p / "correlation.csv").exists()'''

new_key = '''def _key(disease_name: str, min_phase: int = 2, max_candidates: int = 10) -> str:
    """Cache key includes disease + settings, so different settings get separate caches."""
    slug = disease_name.lower().strip().replace(" ", "_")
    settings = f"p{min_phase}_n{max_candidates}"
    return hashlib.md5((slug + settings).encode()).hexdigest()[:12] + "_" + slug[:30] + "_" + settings


def cache_path(disease_name: str, min_phase: int = 2, max_candidates: int = 10) -> Path:
    return CACHE_DIR / f"{_key(disease_name, min_phase, max_candidates)}"


def has_cache(disease_name: str, min_phase: int = 2, max_candidates: int = 10) -> bool:
    p = cache_path(disease_name, min_phase, max_candidates)
    return (p / "candidates.csv").exists() and (p / "correlation.csv").exists()'''

if old_key in content:
    content = content.replace(old_key, new_key)
    cache_file.write_text(content, encoding="utf-8")
    print("✅ Cache key now includes settings")
else:
    print("⚠️  Old _key function not found — checking file...")
    print(content[:800])

✅ Cache key now includes settings


In [20]:
from pathlib import Path

app_file = Path(r"C:\Users\Administrator\RepurposeAlpha\app\app.py")
content = app_file.read_text(encoding="utf-8")

# Patch the load call
old_load = '''    if use_cache and cache_mod.has_cache(disease_input):
        candidates, corr, meta = cache_mod.load_analysis(disease_input)
        st.success(f"✓ Loaded from cache ({meta.get('n_candidates', '?')} candidates)")'''

new_load = '''    if use_cache and cache_mod.has_cache(disease_input, min_phase=min_phase, max_candidates=max_candidates):
        candidates, corr, meta = cache_mod.load_analysis(disease_input, min_phase=min_phase, max_candidates=max_candidates)
        st.success(f"✓ Loaded from cache ({meta.get('n_candidates', '?')} candidates)")'''

if old_load in content:
    content = content.replace(old_load, new_load)
    print("✅ Updated has_cache / load_analysis calls")

# Patch the save call
old_save = '''            cache_mod.save_analysis(disease_input, candidates, corr, metadata={"min_phase": min_phase})'''
new_save = '''            cache_mod.save_analysis(disease_input, candidates, corr,
                                     min_phase=min_phase, max_candidates=max_candidates,
                                     metadata={"min_phase": min_phase, "max_candidates": max_candidates})'''
if old_save in content:
    content = content.replace(old_save, new_save)
    print("✅ Updated save_analysis call")

app_file.write_text(content, encoding="utf-8")
print(f"✅ app.py patched ({len(content):,} bytes)")

✅ Updated has_cache / load_analysis calls
✅ Updated save_analysis call
✅ app.py patched (23,759 bytes)


In [21]:
from pathlib import Path

cache_file = Path(r"C:\Users\Administrator\RepurposeAlpha\src\cache\__init__.py")
content = cache_file.read_text(encoding="utf-8")

# Update save_analysis
old_save = '''def save_analysis(disease_name: str, candidates, correlation, metadata=None):
    """Save candidates + correlation matrix to cache."""
    p = cache_path(disease_name)'''

new_save = '''def save_analysis(disease_name: str, candidates, correlation,
                   min_phase: int = 2, max_candidates: int = 10, metadata=None):
    """Save candidates + correlation matrix to cache."""
    p = cache_path(disease_name, min_phase, max_candidates)'''

if old_save in content:
    content = content.replace(old_save, new_save)
    print("✅ Updated save_analysis signature")

# Update load_analysis
old_load = '''def load_analysis(disease_name: str):
    """Load cached candidates + correlation. Returns (candidates, correlation, metadata)."""
    import pandas as pd
    p = cache_path(disease_name)'''

new_load = '''def load_analysis(disease_name: str, min_phase: int = 2, max_candidates: int = 10):
    """Load cached candidates + correlation. Returns (candidates, correlation, metadata)."""
    import pandas as pd
    p = cache_path(disease_name, min_phase, max_candidates)'''

if old_load in content:
    content = content.replace(old_load, new_load)
    print("✅ Updated load_analysis signature")

cache_file.write_text(content, encoding="utf-8")
print(f"✅ cache/__init__.py saved ({len(content):,} bytes)")

✅ Updated save_analysis signature
✅ Updated load_analysis signature
✅ cache/__init__.py saved (2,821 bytes)


In [22]:
%%writefile C:\Users\Administrator\RepurposeAlpha\src\discovery\__init__.py
"""
RepurposeAlpha — disease to candidates discovery via Open Targets.

Selects the BEST N candidates by:
1. Clinical phase (higher = better)
2. Approval status (bonus)
3. Drug type (biologics preferred for FTO)
4. Data richness proxy
"""
import requests
import pandas as pd
from typing import Optional

OPEN_TARGETS_API = "https://api.platform.opentargets.org/api/v4/graphql"

# Phase → numeric score for ranking (higher = more advanced)
STAGE_RANK = {
    "APPROVAL":   5.0,
    "PHASE_4":    4.5,
    "PHASE_3":    4.0,
    "PHASE_2_3":  3.5,
    "PHASE_2":    3.0,
    "PHASE_1_2":  2.5,
    "PHASE_1":    2.0,
    "PRECLINICAL": 1.0,
    "UNKNOWN":    0.5,
}


def search_disease(disease_name: str) -> Optional[dict]:
    query = """
    query searchDisease($name: String!) {
      search(queryString: $name, entityNames: ["disease"], page: {index: 0, size: 5}) {
        hits { id name entity }
      }
    }
    """
    try:
        r = requests.post(OPEN_TARGETS_API,
                          json={"query": query, "variables": {"name": disease_name}},
                          timeout=30)
        if r.status_code == 200:
            hits = r.json().get("data", {}).get("search", {}).get("hits", [])
            if hits:
                return hits[0]
    except Exception as e:
        print(f"  Search failed: {e}")
    return None


def get_drugs_for_disease(efo_id: str) -> pd.DataFrame:
    """Fetch ALL drugs via drugAndClinicalCandidates — no size limit."""
    query = """
    query diseaseDrugs($efoId: String!) {
      disease(efoId: $efoId) {
        id
        name
        drugAndClinicalCandidates {
          count
          rows {
            maxClinicalStage
            drug {
              id
              name
              drugType
              maximumClinicalStage
              isApproved
            }
          }
        }
      }
    }
    """
    rows = []
    try:
        r = requests.post(OPEN_TARGETS_API,
                          json={"query": query, "variables": {"efoId": efo_id}},
                          timeout=30)
        if r.status_code == 200:
            body = r.json()
            data = body.get("data", {}).get("disease", {})
            if data:
                container = data.get("drugAndClinicalCandidates") or {}
                count = container.get("count", 0)
                for row in container.get("rows", []):
                    drug = row.get("drug", {}) or {}
                    stage = row.get("maxClinicalStage")
                    rows.append({
                        "chembl_id":  (drug.get("id") or "").replace("CHEMBL_", "CHEMBL"),
                        "drug_name":  drug.get("name", ""),
                        "stage_str":  stage,
                        "max_phase":  STAGE_RANK.get(stage, 0.5),
                        "is_approved": bool(drug.get("isApproved")),
                        "drug_type":  drug.get("drugType", ""),
                    })
                print(f"    (API reported {count} total associations)")
    except Exception as e:
        print(f"  Query failed: {e}")
    return pd.DataFrame(rows)


def rank_and_select(df: pd.DataFrame, max_results: int,
                    min_phase: int = 2, verbose: bool = True) -> pd.DataFrame:
    """
    Rank candidates by selection criteria and return the top N.

    Ranking formula:
      0.70 × normalized_phase
    + 0.20 × (1 if approved else 0)
    + 0.10 × (1 if biologic else 0)
    """
    if df.empty:
        return df

    # Filter by minimum phase
    stage_to_num = {
        "APPROVAL": 4.0, "PHASE_4": 4.0, "PHASE_3": 3.0, "PHASE_2_3": 2.5,
        "PHASE_2": 2.0, "PHASE_1_2": 1.5, "PHASE_1": 1.0, "PRECLINICAL": 0.0,
    }
    df = df.copy()
    df["phase_num"] = df["stage_str"].map(stage_to_num).fillna(0)
    df = df[df["phase_num"] >= min_phase].drop_duplicates("chembl_id")

    if df.empty:
        return df

    # Scoring
    df["phase_score"] = df["phase_num"] / 4.0          # normalized to [0, 1]
    df["approved_score"] = df["is_approved"].astype(int)
    df["biologic_score"] = df["drug_type"].str.lower().apply(
        lambda x: 1 if "antibody" in str(x).lower() or "protein" in str(x).lower() or "peptide" in str(x).lower() else 0
    )

    df["rank_score"] = (
        0.70 * df["phase_score"] +
        0.20 * df["approved_score"] +
        0.10 * df["biologic_score"]
    )

    # Sort by rank score descending
    df = df.sort_values("rank_score", ascending=False).reset_index(drop=True)

    if verbose:
        print(f"\n  Ranked {len(df)} candidates. Selection criteria:")
        print(f"    - 70% clinical phase (later = better)")
        print(f"    - 20% already approved")
        print(f"    - 10% biologic (harder to genericize)")
        print(f"\n  Top {min(max_results, len(df))} selected:")

    # Store rationale for UI display
    df["selection_rationale"] = df.apply(
        lambda r: f"Phase {r['stage_str']} · {'approved' if r['is_approved'] else 'investigational'} · score {r['rank_score']:.2f}",
        axis=1,
    )

    return df.head(max_results).reset_index(drop=True)


def get_candidates_for_disease(disease_name: str, min_phase: int = 2,
                                max_results: int = 10,
                                verbose: bool = True) -> pd.DataFrame:
    """Main entry point — returns the BEST N candidates for a disease."""
    print(f"→ Searching Open Targets for '{disease_name}'...")
    disease = search_disease(disease_name)
    if not disease:
        print(f"  ❌ Not found")
        return pd.DataFrame()

    print(f"  ✓ Found: {disease['name']} (EFO: {disease['id']})")
    print(f"→ Fetching all associated drugs...")
    all_drugs = get_drugs_for_disease(disease["id"])

    if all_drugs.empty:
        print(f"  ❌ No drugs returned")
        return pd.DataFrame()

    print(f"  ✓ Retrieved {len(all_drugs)} candidates from Open Targets")

    # Rank and select the best N
    selected = rank_and_select(all_drugs, max_results=max_results,
                                min_phase=min_phase, verbose=verbose)

    return selected

Overwriting C:\Users\Administrator\RepurposeAlpha\src\discovery\__init__.py


In [23]:
import sys
sys.path.insert(0, r"C:\Users\Administrator\RepurposeAlpha\src")
import importlib
import discovery
importlib.reload(discovery)

# Test: ask for top 5 out of ALL candidates for coronary artery
df = discovery.get_candidates_for_disease("coronary artery disease", min_phase=2, max_results=5)
print("\n--- Selected 5 ---")
print(df[["drug_name", "stage_str", "is_approved", "drug_type", "rank_score", "selection_rationale"]].to_string(index=False))

→ Searching Open Targets for 'coronary artery disease'...
  ✓ Found: coronary artery disorder (EFO: MONDO_0005010)
→ Fetching all associated drugs...
  ❌ No drugs returned

--- Selected 5 ---


KeyError: "None of [Index(['drug_name', 'stage_str', 'is_approved', 'drug_type', 'rank_score',\n       'selection_rationale'],\n      dtype='str')] are in the [columns]"

In [24]:
%%writefile C:\Users\Administrator\RepurposeAlpha\src\discovery\__init__.py
"""
RepurposeAlpha — disease to candidates discovery via Open Targets.
Ranked selection: best N by clinical phase + approval + drug type.
"""
import requests
import pandas as pd
from typing import Optional

OPEN_TARGETS_API = "https://api.platform.opentargets.org/api/v4/graphql"

STAGE_RANK = {
    "APPROVAL":    4.0,
    "PHASE_4":     4.0,
    "PHASE_3":     3.0,
    "PHASE_2_3":   2.5,
    "PHASE_2":     2.0,
    "PHASE_1_2":   1.5,
    "PHASE_1":     1.0,
    "PRECLINICAL": 0.5,
    "UNKNOWN":     0.0,
}


def search_disease(disease_name: str) -> Optional[dict]:
    query = """
    query searchDisease($name: String!) {
      search(queryString: $name, entityNames: ["disease"], page: {index: 0, size: 5}) {
        hits { id name entity }
      }
    }
    """
    try:
        r = requests.post(OPEN_TARGETS_API,
                          json={"query": query, "variables": {"name": disease_name}},
                          timeout=30)
        if r.status_code == 200:
            hits = r.json().get("data", {}).get("search", {}).get("hits", [])
            if hits:
                return hits[0]
    except Exception as e:
        print(f"  Search failed: {e}")
    return None


def get_drugs_for_disease(efo_id: str, verbose: bool = True) -> pd.DataFrame:
    """Fetch drugs via drugAndClinicalCandidates. No isApproved — derived from stage."""
    query = """
    query diseaseDrugs($efoId: String!) {
      disease(efoId: $efoId) {
        id
        name
        drugAndClinicalCandidates {
          count
          rows {
            maxClinicalStage
            drug {
              id
              name
              drugType
            }
          }
        }
      }
    }
    """
    rows = []
    try:
        r = requests.post(OPEN_TARGETS_API,
                          json={"query": query, "variables": {"efoId": efo_id}},
                          timeout=30)
        if r.status_code == 200:
            body = r.json()
            if "errors" in body:
                if verbose:
                    print(f"    GraphQL errors: {body['errors'][:1]}")
            data = body.get("data", {}).get("disease", {})
            if data:
                container = data.get("drugAndClinicalCandidates") or {}
                count = container.get("count", 0)
                if verbose:
                    print(f"    (API reported {count} total associations)")
                for row in container.get("rows", []):
                    drug = row.get("drug", {}) or {}
                    stage = row.get("maxClinicalStage") or "UNKNOWN"
                    rows.append({
                        "chembl_id":   (drug.get("id") or "").replace("CHEMBL_", "CHEMBL"),
                        "drug_name":   drug.get("name", ""),
                        "stage_str":   stage,
                        "max_phase":   STAGE_RANK.get(stage, 0.0),
                        "is_approved": stage in ("APPROVAL", "PHASE_4"),
                        "drug_type":   drug.get("drugType", ""),
                    })
        else:
            if verbose:
                print(f"    HTTP {r.status_code}")
    except Exception as e:
        if verbose:
            print(f"  Query failed: {e}")
    return pd.DataFrame(rows)


def rank_and_select(df: pd.DataFrame, max_results: int,
                    min_phase: int = 2, verbose: bool = True) -> pd.DataFrame:
    """
    Rank and select top N candidates.
    Scoring: 70% phase + 20% approved + 10% biologic.
    """
    if df.empty:
        return df

    df = df.copy()
    df["phase_num"] = df["stage_str"].map(STAGE_RANK).fillna(0.0)

    # Filter by min_phase
    stage_to_phase = {
        "APPROVAL": 4.0, "PHASE_4": 4.0, "PHASE_3": 3.0, "PHASE_2_3": 2.5,
        "PHASE_2": 2.0, "PHASE_1_2": 1.5, "PHASE_1": 1.0, "PRECLINICAL": 0.5,
        "UNKNOWN": 0.0,
    }
    df["phase_for_filter"] = df["stage_str"].map(stage_to_phase).fillna(0.0)
    df = df[df["phase_for_filter"] >= min_phase].drop_duplicates("chembl_id")

    if df.empty:
        return df

    # Scoring
    df["phase_score"] = (df["phase_for_filter"] / 4.0).clip(0, 1)
    df["approved_score"] = df["is_approved"].astype(int)
    df["biologic_score"] = df["drug_type"].str.lower().apply(
        lambda x: 1 if any(k in str(x).lower() for k in ["antibody", "protein", "peptide"]) else 0
    )

    df["rank_score"] = (
        0.70 * df["phase_score"] +
        0.20 * df["approved_score"] +
        0.10 * df["biologic_score"]
    )

    df = df.sort_values("rank_score", ascending=False).reset_index(drop=True)

    df["selection_rationale"] = df.apply(
        lambda r: f"Phase {r['stage_str']} · {'approved' if r['is_approved'] else 'investigational'} · score {r['rank_score']:.2f}",
        axis=1,
    )

    if verbose:
        print(f"\n  Ranked {len(df)} candidates. Selection criteria:")
        print(f"    - 70% clinical phase · 20% approved · 10% biologic")

    return df.head(max_results).reset_index(drop=True)


def get_candidates_for_disease(disease_name: str, min_phase: int = 2,
                                max_results: int = 10,
                                verbose: bool = True) -> pd.DataFrame:
    print(f"→ Searching Open Targets for '{disease_name}'...")
    disease = search_disease(disease_name)
    if not disease:
        print(f"  ❌ Not found")
        return pd.DataFrame()

    print(f"  ✓ Found: {disease['name']} (EFO: {disease['id']})")
    print(f"→ Fetching all associated drugs...")
    all_drugs = get_drugs_for_disease(disease["id"], verbose=verbose)

    if all_drugs.empty:
        print(f"  ❌ No drugs returned")
        return pd.DataFrame()

    print(f"  ✓ Retrieved {len(all_drugs)} candidates from Open Targets")
    return rank_and_select(all_drugs, max_results=max_results,
                            min_phase=min_phase, verbose=verbose)

Overwriting C:\Users\Administrator\RepurposeAlpha\src\discovery\__init__.py


In [25]:
import sys
sys.path.insert(0, r"C:\Users\Administrator\RepurposeAlpha\src")
import importlib
import discovery
importlib.reload(discovery)

# Test with malaria — we know this works
print("=" * 70)
print("TEST 1: MALARIA (Max=5)")
print("=" * 70)
df = discovery.get_candidates_for_disease("malaria", min_phase=2, max_results=5)

if df.empty:
    print("❌ Empty DataFrame")
else:
    print(f"\n✅ Selected {len(df)} candidates\n")
    print(df[["drug_name", "stage_str", "is_approved", "rank_score", "selection_rationale"]].to_string(index=False))

TEST 1: MALARIA (Max=5)
→ Searching Open Targets for 'malaria'...
  ✓ Found: malaria (EFO: MONDO_0005136)
→ Fetching all associated drugs...
    (API reported 102 total associations)
  ✓ Retrieved 102 candidates from Open Targets

  Ranked 79 candidates. Selection criteria:
    - 70% clinical phase · 20% approved · 10% biologic

✅ Selected 5 candidates

                 drug_name stage_str  is_approved  rank_score                    selection_rationale
      PRIMAQUINE PHOSPHATE  APPROVAL         True         0.9 Phase APPROVAL · approved · score 0.90
PIPERAQUINE TETRAPHOSPHATE  APPROVAL         True         0.9 Phase APPROVAL · approved · score 0.90
              SULFADIAZINE  APPROVAL         True         0.9 Phase APPROVAL · approved · score 0.90
                 PROGUANIL  APPROVAL         True         0.9 Phase APPROVAL · approved · score 0.90
              HALOFANTRINE  APPROVAL         True         0.9 Phase APPROVAL · approved · score 0.90


In [26]:
from pathlib import Path

f = Path(r"C:\Users\Administrator\RepurposeAlpha\src\discovery\__init__.py")
content = f.read_text(encoding="utf-8")

# Add a diversity-aware selection function at the end
addition = '''

def select_diverse_top_n(df: pd.DataFrame, n: int,
                          rank_col: str = "rank_score",
                          candidate_pool_multiplier: int = 3,
                          verbose: bool = True) -> pd.DataFrame:
    """
    Two-stage selection:
    1. Take top (n × multiplier) by rank_score
    2. From that pool, greedily pick n candidates that are least correlated with each other

    Correlation is computed via the existing correlation engine on the candidate pool.
    """
    if df.empty or len(df) <= n:
        return df

    # Stage 1: get the top pool by rank
    pool_size = min(len(df), n * candidate_pool_multiplier)
    pool = df.head(pool_size).copy()

    if verbose:
        print(f"  Stage 1: selected top {pool_size} by rank score")

    # Stage 2: diversity selection using correlation
    try:
        import correlation as corr_mod
        corr_matrix = corr_mod.build_correlation_matrix(
            pool["chembl_id"].tolist(), verbose=False
        )
        if corr_matrix is None or corr_matrix.empty:
            if verbose:
                print("  Stage 2: correlation failed — falling back to rank-only")
            return df.head(n).reset_index(drop=True)

        # Greedy diversity pick
        selected = []
        remaining = pool["chembl_id"].tolist()

        # Always take the highest-ranked first
        first = remaining.pop(0)
        selected.append(first)

        while len(selected) < n and remaining:
            # For each remaining candidate, compute max correlation to any selected
            best_cid = None
            best_max_corr = float("inf")

            for cid in remaining:
                if cid not in corr_matrix.index:
                    continue
                max_corr = max(abs(corr_matrix.loc[cid, s]) for s in selected if s in corr_matrix.columns)
                if max_corr < best_max_corr:
                    best_max_corr = max_corr
                    best_cid = cid

            if best_cid is None:
                break

            selected.append(best_cid)
            remaining.remove(best_cid)

            if verbose:
                print(f"    + added {best_cid} (max corr to selected: {best_max_corr:.2f})")

        # Reorder the pool to match selection order, then return
        result = pool[pool["chembl_id"].isin(selected)].copy()
        result["_order"] = result["chembl_id"].apply(lambda x: selected.index(x))
        result = result.sort_values("_order").drop(columns=["_order"])

        if verbose:
            print(f"  Stage 2: selected {len(result)} diverse candidates")

        return result.reset_index(drop=True)

    except Exception as e:
        if verbose:
            print(f"  Stage 2 failed: {e} — falling back to rank-only")
        return df.head(n).reset_index(drop=True)
'''

if "select_diverse_top_n" not in content:
    content += addition
    f.write_text(content, encoding="utf-8")
    print("✅ Added select_diverse_top_n function")
else:
    print("ℹ️  Already present")

✅ Added select_diverse_top_n function


In [27]:
from pathlib import Path

f = Path(r"C:\Users\Administrator\RepurposeAlpha\src\discovery\__init__.py")
content = f.read_text(encoding="utf-8")

old = '''    print(f"  ✓ Retrieved {len(all_drugs)} candidates from Open Targets")
    return rank_and_select(all_drugs, max_results=max_results,
                            min_phase=min_phase, verbose=verbose)'''

new = '''    print(f"  ✓ Retrieved {len(all_drugs)} candidates from Open Targets")

    # Rank by phase + approval + type
    ranked = rank_and_select(all_drugs, max_results=len(all_drugs),
                              min_phase=min_phase, verbose=verbose)

    if ranked.empty:
        return ranked

    # Then select a diverse top-N
    if len(ranked) > max_results:
        print(f"→ Selecting {max_results} most diverse from top candidates...")
        return select_diverse_top_n(ranked, n=max_results, verbose=verbose)
    else:
        return ranked.head(max_results).reset_index(drop=True)'''

if old in content:
    content = content.replace(old, new)
    f.write_text(content, encoding="utf-8")
    print("✅ Wired diversity selection into main entry point")
else:
    print("⚠️  Marker not found — check the file")

✅ Wired diversity selection into main entry point


In [28]:
import sys
sys.path.insert(0, r"C:\Users\Administrator\RepurposeAlpha\src")
import importlib
import discovery, correlation
importlib.reload(correlation)
importlib.reload(discovery)

print("=" * 70)
print("MALARIA — Max=5 with DIVERSITY selection")
print("=" * 70)
df = discovery.get_candidates_for_disease("malaria", min_phase=2, max_results=5, verbose=True)

if not df.empty:
    print(f"\n✅ Final selection ({len(df)} candidates):\n")
    print(df[["drug_name", "stage_str", "rank_score", "selection_rationale"]].to_string(index=False))

MALARIA — Max=5 with DIVERSITY selection
→ Searching Open Targets for 'malaria'...
  ✓ Found: malaria (EFO: MONDO_0005136)
→ Fetching all associated drugs...
    (API reported 102 total associations)
  ✓ Retrieved 102 candidates from Open Targets

  Ranked 79 candidates. Selection criteria:
    - 70% clinical phase · 20% approved · 10% biologic
→ Selecting 5 most diverse from top candidates...
  Stage 1: selected top 15 by rank score
    + added CHEMBL4297927 (max corr to selected: 0.00)
    + added CHEMBL979 (max corr to selected: 0.01)
    + added CHEMBL1201059 (max corr to selected: 0.01)
    + added CHEMBL244800 (max corr to selected: 0.01)
  Stage 2: selected 5 diverse candidates

✅ Final selection (5 candidates):

                 drug_name stage_str  rank_score                    selection_rationale
      PRIMAQUINE PHOSPHATE  APPROVAL         0.9 Phase APPROVAL · approved · score 0.90
LUMEFANTRINE CO-ARTEMETHER  APPROVAL         0.9 Phase APPROVAL · approved · score 0.90
      

In [29]:
import sys
sys.path.insert(0, r"C:\Users\Administrator\RepurposeAlpha\src")
import importlib
import discovery, correlation
importlib.reload(correlation)
importlib.reload(discovery)

# Test just the diversity function on a small set
import pandas as pd

# Take a known small pool
pool = pd.DataFrame({
    "chembl_id": ["CHEMBL43128", "CHEMBL1652442", "CHEMBL439", "CHEMBL1483", "CHEMBL112"],
    "drug_name": ["PRIMAQUINE PHOSPHATE", "PIPERAQUINE TETRAPHOSPHATE", "SULFADIAZINE", "ALBENDAZOLE", "ACETAMINOPHEN"],
    "stage_str": ["APPROVAL"] * 5,
    "rank_score": [0.9, 0.9, 0.9, 0.9, 0.9],
})

print("Testing correlation matrix call on 5 drugs...")
try:
    corr = correlation.build_correlation_matrix(pool["chembl_id"].tolist(), verbose=True)
    if corr is not None and not corr.empty:
        print(f"\n✅ Correlation matrix built: {corr.shape}")
        print(corr.round(2))
    else:
        print("\n❌ Correlation returned empty")
except Exception as e:
    import traceback
    print(f"\n❌ Error: {e}")
    traceback.print_exc()

Testing correlation matrix call on 5 drugs...
  Fetching 5 drugs (parallel)...
  Data availability:
    targets:  3/5
    ATC:      3/5
    MoA:      4/5
    SMILES:   5/5

  Entropy-weighted signal combination:
  Entropy analysis (variance floor = 0.0010):
  signal         entropy    variance    weight
    structure        0.141     0.00142     1.000
    target           0.000     0.00000     0.000  (excluded)
    affinity         0.000     0.00000     0.000  (excluded)
    scaffold         0.000     0.00000     0.000  (excluded)
    moa              0.000     0.00000     0.000  (excluded)
    atc              0.000     0.00000     0.000  (excluded)
    assay            0.000     0.00064     0.000  (excluded)
  Total weight: 1.000

  ✓ Composite matrix built (entropy-weighted)

✅ Correlation matrix built: (5, 5)
               CHEMBL43128  CHEMBL1652442  CHEMBL439  CHEMBL1483  CHEMBL112
CHEMBL43128           1.00           0.19       0.19        0.20       0.11
CHEMBL1652442         0

In [30]:
from pathlib import Path
import re

f = Path(r"C:\Users\Administrator\RepurposeAlpha\src\discovery\__init__.py")
content = f.read_text(encoding="utf-8")

new_func = '''def select_diverse_top_n(df: pd.DataFrame, n: int,
                          rank_col: str = "rank_score",
                          verbose: bool = True) -> pd.DataFrame:
    """
    Fast diversity selection using drug-name family signal.

    Strategy:
    1. Always take the top-ranked candidate.
    2. Subsequent picks prefer candidates whose first word (family)
       isn't already represented.
    3. Fall back to rank order if not enough unique families.
    """
    if df.empty or len(df) <= n:
        return df

    def family(name: str) -> str:
        first = str(name).strip().upper().split()[0] if name else "UNKNOWN"
        return first

    df = df.copy()
    df["_family"] = df["drug_name"].apply(family)

    selected_rows = []
    seen_families = set()

    # Pass 1: one per unique family
    for _, row in df.iterrows():
        if row["_family"] not in seen_families:
            selected_rows.append(row)
            seen_families.add(row["_family"])
            if len(selected_rows) >= n:
                break

    # Pass 2: fill remaining slots by rank
    if len(selected_rows) < n:
        selected_ids = {r["chembl_id"] for r in selected_rows}
        for _, row in df.iterrows():
            if row["chembl_id"] not in selected_ids:
                selected_rows.append(row)
                if len(selected_rows) >= n:
                    break

    result = pd.DataFrame(selected_rows).drop(columns=["_family"]).reset_index(drop=True)

    if verbose:
        print(f"  Diversity selection: {len(result)} from {len(df)} pool")

    return result
'''

# Find the function and replace it
start = content.find("def select_diverse_top_n")
if start > 0:
    # Find the next top-level def after this
    next_def = content.find("\ndef ", start + 10)
    if next_def == -1:
        next_def = len(content)
    content = content[:start] + new_func + "\n" + content[next_def:]
    f.write_text(content, encoding="utf-8")
    print("✅ Replaced with fast heuristic")
else:
    print("⚠️  Function not found")

✅ Replaced with fast heuristic


In [31]:
import sys
sys.path.insert(0, r"C:\Users\Administrator\RepurposeAlpha\src")
import importlib
import discovery
importlib.reload(discovery)

print("=" * 70)
print("MALARIA — Max=5 with FAST diversity")
print("=" * 70)
df = discovery.get_candidates_for_disease("malaria", min_phase=2, max_results=5)
if not df.empty:
    print(f"\n✅ Final selection:")
    print(df[["drug_name", "stage_str", "rank_score"]].to_string(index=False))

MALARIA — Max=5 with FAST diversity
→ Searching Open Targets for 'malaria'...
  ✓ Found: malaria (EFO: MONDO_0005136)
→ Fetching all associated drugs...
    (API reported 102 total associations)
  ✓ Retrieved 102 candidates from Open Targets

  Ranked 79 candidates. Selection criteria:
    - 70% clinical phase · 20% approved · 10% biologic
→ Selecting 5 most diverse from top candidates...
  Diversity selection: 5 from 79 pool

✅ Final selection:
                 drug_name stage_str  rank_score
      PRIMAQUINE PHOSPHATE  APPROVAL         0.9
PIPERAQUINE TETRAPHOSPHATE  APPROVAL         0.9
              SULFADIAZINE  APPROVAL         0.9
                 PROGUANIL  APPROVAL         0.9
              HALOFANTRINE  APPROVAL         0.9


In [32]:
from pathlib import Path

f = Path(r"C:\Users\Administrator\RepurposeAlpha\src\discovery\__init__.py")
content = f.read_text(encoding="utf-8")

new_func = '''def select_diverse_top_n(df: pd.DataFrame, n: int,
                          rank_col: str = "rank_score",
                          verbose: bool = True) -> pd.DataFrame:
    """
    Phase-stratified diversity selection.

    Strategy:
    1. Bucket candidates by clinical stage (approved / phase 3 / phase 2 / phase 1)
    2. Pick from each bucket in round-robin fashion until n selected
    3. Within each bucket, prefer unique drug-name families

    This gives a portfolio spread across the pipeline, not 5 approved drugs.
    """
    if df.empty or len(df) <= n:
        return df

    df = df.copy()
    df["_family"] = df["drug_name"].apply(
        lambda x: str(x).strip().upper().split()[0] if x else "UNKNOWN"
    )

    # Bucket by stage (approved highest priority)
    def bucket(stage_str):
        s = str(stage_str).upper()
        if s in ("APPROVAL", "PHASE_4"):
            return "approved"
        if s == "PHASE_3":
            return "phase_3"
        if s in ("PHASE_2_3", "PHASE_2"):
            return "phase_2"
        return "phase_1"

    df["_bucket"] = df["stage_str"].apply(bucket)

    # Round-robin pick across buckets
    bucket_order = ["approved", "phase_3", "phase_2", "phase_1"]
    buckets = {b: df[df["_bucket"] == b].to_dict("records") for b in bucket_order}

    selected_rows = []
    selected_ids = set()
    seen_families = set()

    # Round 1: one per bucket
    for b in bucket_order:
        for row in buckets.get(b, []):
            if row["_family"] not in seen_families and row["chembl_id"] not in selected_ids:
                selected_rows.append(row)
                selected_ids.add(row["chembl_id"])
                seen_families.add(row["_family"])
                break

    # Round 2+: fill remaining slots by rank order across all buckets
    if len(selected_rows) < n:
        for b in bucket_order:
            for row in buckets.get(b, []):
                if row["chembl_id"] not in selected_ids:
                    selected_rows.append(row)
                    selected_ids.add(row["chembl_id"])
                    if len(selected_rows) >= n:
                        break
            if len(selected_rows) >= n:
                break

    result = pd.DataFrame(selected_rows).drop(columns=["_family", "_bucket"]).reset_index(drop=True)

    if verbose:
        print(f"  Phase-stratified selection: {len(result)} candidates")
        for _, r in result.iterrows():
            print(f"    • {r['drug_name'][:35]:35s}  ({r['stage_str']})")

    return result
'''

start = content.find("def select_diverse_top_n")
if start > 0:
    next_def = content.find("\ndef ", start + 10)
    if next_def == -1:
        next_def = len(content)
    content = content[:start] + new_func + "\n" + content[next_def:]
    f.write_text(content, encoding="utf-8")
    print("✅ Patched: phase-stratified selection")
else:
    print("⚠️  Function not found")

✅ Patched: phase-stratified selection


In [33]:
import sys
sys.path.insert(0, r"C:\Users\Administrator\RepurposeAlpha\src")
import importlib
import discovery
importlib.reload(discovery)

print("=" * 70)
print("MALARIA — Max=5 with PHASE-STRATIFIED selection")
print("=" * 70)
df = discovery.get_candidates_for_disease("malaria", min_phase=2, max_results=5)

if not df.empty:
    print(f"\n✅ Final selection:")
    print(df[["drug_name", "stage_str", "rank_score"]].to_string(index=False))

MALARIA — Max=5 with PHASE-STRATIFIED selection
→ Searching Open Targets for 'malaria'...
  ✓ Found: malaria (EFO: MONDO_0005136)
→ Fetching all associated drugs...
    (API reported 102 total associations)
  ✓ Retrieved 102 candidates from Open Targets

  Ranked 79 candidates. Selection criteria:
    - 70% clinical phase · 20% approved · 10% biologic
→ Selecting 5 most diverse from top candidates...
  Phase-stratified selection: 5 candidates
    • PRIMAQUINE PHOSPHATE                 (APPROVAL)
    • INSULIN GLARGINE                     (PHASE_3)
    • EPOETIN BETA                         (PHASE_2_3)
    • PIPERAQUINE TETRAPHOSPHATE           (APPROVAL)
    • SULFADIAZINE                         (APPROVAL)

✅ Final selection:
                 drug_name stage_str  rank_score
      PRIMAQUINE PHOSPHATE  APPROVAL      0.9000
          INSULIN GLARGINE   PHASE_3      0.6250
              EPOETIN BETA PHASE_2_3      0.5375
PIPERAQUINE TETRAPHOSPHATE  APPROVAL      0.9000
              SULF

In [34]:
from pathlib import Path

f = Path(r"C:\Users\Administrator\RepurposeAlpha\src\discovery\__init__.py")
content = f.read_text(encoding="utf-8")

# 1. Update the query to include the association score
old_query = '''    query = """
    query diseaseDrugs($efoId: String!) {
      disease(efoId: $efoId) {
        id
        name
        drugAndClinicalCandidates {
          count
          rows {
            maxClinicalStage
            drug {
              id
              name
              drugType
            }
          }
        }
      }
    }
    """'''

new_query = '''    query = """
    query diseaseDrugs($efoId: String!) {
      disease(efoId: $efoId) {
        id
        name
        drugAndClinicalCandidates {
          count
          rows {
            maxClinicalStage
            drug {
              id
              name
              drugType
            }
          }
        }
      }
    }
    """'''

# Actually the schema doesn't have a per-row score field. Let's not add it.
# Instead, revert selection to: rank by phase, then pick diverse names
# (which is what gave the good output before)

new_func = '''def select_diverse_top_n(df: pd.DataFrame, n: int,
                          rank_col: str = "rank_score",
                          verbose: bool = True) -> pd.DataFrame:
    """
    Rank-first diversity selection.

    1. Sort by rank_score (phase + approved + biologic).
    2. Prefer candidates with drug-name families not already picked.
    3. Fill remaining slots by rank order.

    This avoids 5 variants of the same drug while preserving the
    "most clinically advanced first" ordering.
    """
    if df.empty or len(df) <= n:
        return df

    df = df.copy()
    df["_family"] = df["drug_name"].apply(
        lambda x: str(x).strip().upper().split()[0] if x else "UNKNOWN"
    )

    selected_rows = []
    seen_families = set()
    selected_ids = set()

    # Pass 1: pick one per unique family in rank order
    for _, row in df.iterrows():
        if row["_family"] not in seen_families and row["chembl_id"] not in selected_ids:
            selected_rows.append(row)
            seen_families.add(row["_family"])
            selected_ids.add(row["chembl_id"])
            if len(selected_rows) >= n:
                break

    # Pass 2: fill remaining slots by rank
    if len(selected_rows) < n:
        for _, row in df.iterrows():
            if row["chembl_id"] not in selected_ids:
                selected_rows.append(row)
                selected_ids.add(row["chembl_id"])
                if len(selected_rows) >= n:
                    break

    result = pd.DataFrame(selected_rows).drop(columns=["_family"]).reset_index(drop=True)

    if verbose:
        print(f"  Rank-first diversity: {len(result)} from {len(df)} pool")

    return result
'''

start = content.find("def select_diverse_top_n")
if start > 0:
    next_def = content.find("\ndef ", start + 10)
    if next_def == -1:
        next_def = len(content)
    content = content[:start] + new_func + "\n" + content[next_def:]
    f.write_text(content, encoding="utf-8")
    print("✅ Reverted to rank-first diversity selection")
else:
    print("⚠️  Function not found")

✅ Reverted to rank-first diversity selection


In [35]:
import sys
sys.path.insert(0, r"C:\Users\Administrator\RepurposeAlpha\src")
import importlib
import discovery
importlib.reload(discovery)

df = discovery.get_candidates_for_disease("malaria", min_phase=2, max_results=5)
if not df.empty:
    print(f"\n✅ Final selection:")
    print(df[["drug_name", "stage_str", "rank_score"]].to_string(index=False))

→ Searching Open Targets for 'malaria'...
  ✓ Found: malaria (EFO: MONDO_0005136)
→ Fetching all associated drugs...
    (API reported 102 total associations)
  ✓ Retrieved 102 candidates from Open Targets

  Ranked 79 candidates. Selection criteria:
    - 70% clinical phase · 20% approved · 10% biologic
→ Selecting 5 most diverse from top candidates...
  Rank-first diversity: 5 from 79 pool

✅ Final selection:
                 drug_name stage_str  rank_score
      PRIMAQUINE PHOSPHATE  APPROVAL         0.9
PIPERAQUINE TETRAPHOSPHATE  APPROVAL         0.9
              SULFADIAZINE  APPROVAL         0.9
                 PROGUANIL  APPROVAL         0.9
              HALOFANTRINE  APPROVAL         0.9


In [36]:
import subprocess
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\Administrator\RepurposeAlpha")

def run(cmd, cwd=PROJECT_ROOT):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout.strip() or r.stderr.strip())

print("--- status ---")
print(run("git status --short"))

print("\n--- staging ---")
print(run("git add ."))

print("\n--- committing ---")
print(run('git commit -m "fix: rank-first diverse candidate selection + cache keys + exec narrative"'))

print("\n--- last commit ---")
print(run("git log --oneline -1"))

--- status ---
M app/app.py
 M src/cache/__init__.py
 M src/discovery/__init__.py
 M src/executive/__init__.py

--- staging ---


--- committing ---
[main d992a95] fix: rank-first diverse candidate selection + cache keys + exec narrative
 4 files changed, 178 insertions(+), 56 deletions(-)

--- last commit ---
d992a95 fix: rank-first diverse candidate selection + cache keys + exec narrative


In [37]:
from pathlib import Path

f = Path(r"C:\Users\Administrator\RepurposeAlpha\src\executive\__init__.py")
content = f.read_text(encoding="utf-8")

# Find and replace the verdict logic
old_verdict = '''    # ---------- Overall verdict ----------
    n_neg = sum(1 for c in cards if c.status == "negative")
    n_pos = sum(1 for c in cards if c.status == "positive")
    n_cau = sum(1 for c in cards if c.status == "caution")

    if n_neg >= 2:
        verdict, color = "NOT RECOMMENDED", "red"
    elif n_neg == 1 or n_cau >= 3:
        verdict, color = "CAUTION", "yellow"
    elif n_pos >= 4:
        verdict, color = "RECOMMENDED", "green"
    else:
        verdict, color = "REVIEW", "yellow"'''

new_verdict = '''    # ---------- Overall verdict (conservative) ----------
    n_neg = sum(1 for c in cards if c.status == "negative")
    n_pos = sum(1 for c in cards if c.status == "positive")
    n_cau = sum(1 for c in cards if c.status == "caution")

    # Real Options is a HARD veto — if it says don't start, we can't recommend
    ro_negative = (ro_result is not None) and (ro_result.option_value <= 0)

    if ro_negative and n_neg >= 1:
        # Real Options veto + another feature negative = strong NO
        verdict, color = "NOT RECOMMENDED", "red"
    elif ro_negative:
        # Real Options veto alone = CAUTION (portfolio may still be useful)
        verdict, color = "CAUTION", "yellow"
    elif n_neg >= 2:
        verdict, color = "NOT RECOMMENDED", "red"
    elif n_neg == 1 or n_cau >= 3:
        verdict, color = "CAUTION", "yellow"
    elif n_pos >= 5:
        verdict, color = "RECOMMENDED", "green"
    else:
        verdict, color = "REVIEW", "yellow"'''

if old_verdict in content:
    content = content.replace(old_verdict, new_verdict)
    f.write_text(content, encoding="utf-8")
    print("✅ Verdict logic updated (Real Options as hard veto)")
else:
    print("⚠️  Marker not found — checking...")
    idx = content.find("Overall verdict")
    print(content[max(0, idx-100):idx+400])

✅ Verdict logic updated (Real Options as hard veto)


In [38]:
from pathlib import Path

app_file = Path(r"C:\Users\Administrator\RepurposeAlpha\app\app.py")
content = app_file.read_text(encoding="utf-8")

# Fix the Strategic tab's Real Options slider to use the same estimate
old = '''        peak = st.number_input("Peak sales ($M)", 50, 2000, 200, step=50) * 1e6
        vol  = st.slider("Volatility", 0.20, 0.80, 0.40, 0.05)'''

new = '''        _default_peak = int(executive.estimate_peak_sales(disease) / 1e6)
        _default_vol  = executive.estimate_volatility(disease)
        peak = st.number_input("Peak sales ($M)", 50, 2000, _default_peak, step=50) * 1e6
        vol  = st.slider("Volatility", 0.20, 0.80, _default_vol, 0.05)
        st.caption(f"Default estimated from disease category: {disease}")'''

if old in content:
    content = content.replace(old, new)
    app_file.write_text(content, encoding="utf-8")
    print("✅ Strategic tab peak-sales default now matches Executive Summary")
else:
    # Try alternative spacing
    import re
    new_content = re.sub(
        r'peak = st\.number_input\("Peak sales \(\$M\)", 50, 2000, 200, step=50\) \* 1e6',
        '_default_peak = int(executive.estimate_peak_sales(disease) / 1e6)\n        peak = st.number_input("Peak sales ($M)", 50, 2000, _default_peak, step=50) * 1e6',
        content,
    )
    if new_content != content:
        app_file.write_text(new_content, encoding="utf-8")
        print("✅ Patched with regex")
    else:
        print("⚠️  Could not find slider line. Searching...")
        idx = content.find("Peak sales")
        print(content[max(0, idx-200):idx+300])

✅ Patched with regex


In [39]:
from pathlib import Path

app_file = Path(r"C:\Users\Administrator\RepurposeAlpha\app\app.py")
content = app_file.read_text(encoding="utf-8")

# Add a caption under the verdict banner explaining what it means
old = '''        <div style="font-size:15px; color:#cbd5e1;">{summary.one_liner}</div>
        </div>""", unsafe_allow_html=True)'''

new = '''        <div style="font-size:15px; color:#cbd5e1;">{summary.one_liner}</div>
        </div>""", unsafe_allow_html=True)

    st.caption(
        "Verdict considers all seven features. **Real Options is a hard veto** — "
        "if the project doesn't clear cost of capital, verdict cannot be RECOMMENDED. "
        "Adjust assumptions in the Strategic tab to see how the verdict changes."
    )'''

if old in content and "Verdict considers all seven features" not in content:
    content = content.replace(old, new)
    app_file.write_text(content, encoding="utf-8")
    print("✅ Added verdict explanation caption")
else:
    print("ℹ️  Already added or marker not found")

✅ Added verdict explanation caption


In [40]:
from pathlib import Path
import re

app_file = Path(r"C:\Users\Administrator\RepurposeAlpha\app\app.py")
content = app_file.read_text(encoding="utf-8")

# Find the problem area
idx = content.find("_default_peak")
if idx > 0:
    print("--- Context around the issue ---")
    print(content[max(0, idx-400):idx+400])
    print("--- End context ---")
else:
    print("ℹ️  No _default_peak found. Searching for 'Peak sales'...")
    idx = content.find("Peak sales")
    if idx > 0:
        print(content[max(0, idx-300):idx+400])

--- Context around the issue ---
cept Exception as e:
                st.error(f"Solver error: {e}")


# ---------- TAB 3: STRATEGIC ----------
with tabs[2]:
    sub = st.tabs(["⏳ Real Options", "⚖️ FTO", "🧪 Trials", "💰 Pricing"])

    with sub[0]:
        st.markdown("**When should you kill this project?** Each candidate is modeled as a series of phase gates.")
        c1_, c2_ = st.columns([1, 2])
        with c1_:
            _default_peak = int(executive.estimate_peak_sales(disease) / 1e6)
        peak = st.number_input("Peak sales ($M)", 50, 2000, _default_peak, step=50) * 1e6
            vol  = st.slider("Volatility", 0.20, 0.80, 0.40, 0.05)
        _ph = [
            options.Phase("Phase I", 2.0, float(assumptions.get_value("cost.phase_1_usd", 10e6)), float(assumptions.get_value("pos.phase_1", 0.63))),
           
--- End context ---


In [41]:
from pathlib import Path
import re

app_file = Path(r"C:\Users\Administrator\RepurposeAlpha\app\app.py")
content = app_file.read_text(encoding="utf-8")

# The broken block (mixed indentation)
broken = '''        with c1_:
            _default_peak = int(executive.estimate_peak_sales(disease) / 1e6)
        peak = st.number_input("Peak sales ($M)", 50, 2000, _default_peak, step=50) * 1e6
            vol  = st.slider("Volatility", 0.20, 0.80, 0.40, 0.05)'''

fixed = '''        with c1_:
            _default_peak = int(executive.estimate_peak_sales(disease) / 1e6)
            _default_vol  = executive.estimate_volatility(disease)
            peak = st.number_input("Peak sales ($M)", 50, 2000, _default_peak, step=50) * 1e6
            vol  = st.slider("Volatility", 0.20, 0.80, _default_vol, 0.05)
            st.caption(f"Estimated from disease category: {disease}")'''

if broken in content:
    content = content.replace(broken, fixed)
    app_file.write_text(content, encoding="utf-8")
    print("✅ Fixed indentation")
else:
    # Try a regex-based fix
    pattern = re.compile(
        r'        with c1_:\n'
        r'            _default_peak = int\(executive\.estimate_peak_sales\(disease\) / 1e6\)\n'
        r'        peak = st\.number_input\("Peak sales \(\$M\)", 50, 2000, _default_peak, step=50\) \* 1e6\n'
        r'            vol  = st\.slider\("Volatility", 0\.20, 0\.80, 0\.40, 0\.05\)',
        re.MULTILINE,
    )
    new_content = pattern.sub(fixed, content)
    if new_content != content:
        app_file.write_text(new_content, encoding="utf-8")
        print("✅ Fixed with regex")
    else:
        print("⚠️  Neither approach matched. Running diagnostic...")
        lines = content.splitlines()
        for i in range(315, min(325, len(lines))):
            print(f"{i+1:4d}: {repr(lines[i])}")

✅ Fixed indentation


In [42]:
import ast
from pathlib import Path

app_file = Path(r"C:\Users\Administrator\RepurposeAlpha\app\app.py")
try:
    ast.parse(app_file.read_text(encoding="utf-8"))
    print("✅ File parses correctly — no syntax errors")
except SyntaxError as e:
    print(f"❌ Syntax error at line {e.lineno}: {e.msg}")
    print(f"   Context: {e.text}")

✅ File parses correctly — no syntax errors


In [43]:
import subprocess
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\Administrator\RepurposeAlpha")

def run(cmd, cwd=PROJECT_ROOT):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout.strip() or r.stderr.strip())

print(run("git add ."))
print(run('git commit -m "fix: verdict logic (RO veto) + disease-aware peak-sales consistency"'))
print("\n⚠️  Push from PowerShell.")


[main 5972cc8] fix: verdict logic (RO veto) + disease-aware peak-sales consistency
 2 files changed, 23 insertions(+), 5 deletions(-)

⚠️  Push from PowerShell.


In [44]:
from pathlib import Path
from datetime import datetime

# Where notebooks usually live
search_dirs = [
    Path(r"C:\Users\Administrator"),
    Path(r"C:\Users\Administrator\RepurposeAlpha"),
    Path(r"C:\Users\Administrator\Documents"),
]

notebooks = []
for d in search_dirs:
    if d.exists():
        for nb in d.glob("*.ipynb"):
            # Skip checkpoints
            if ".ipynb_checkpoints" in str(nb):
                continue
            stat = nb.stat()
            notebooks.append({
                "path": nb,
                "name": nb.name,
                "size_kb": round(stat.st_size / 1024, 1),
                "modified": datetime.fromtimestamp(stat.st_mtime).strftime("%Y-%m-%d %H:%M"),
            })

# Sort by modified date
notebooks.sort(key=lambda x: x["modified"], reverse=True)

print(f"Found {len(notebooks)} notebook(s):\n")
for i, nb in enumerate(notebooks, 1):
    print(f"  {i:2d}. {nb['name']:40s}  {nb['size_kb']:>8} KB   {nb['modified']}")
    print(f"      → {nb['path']}")

Found 12 notebook(s):

   1. Untitled8.ipynb                              202.6 KB   2026-09-29 11:21
      → C:\Users\Administrator\Untitled8.ipynb
   2. Untitled7.ipynb                               53.4 KB   2026-09-28 16:44
      → C:\Users\Administrator\Untitled7.ipynb
   3. Untitled5.ipynb                              661.2 KB   2026-09-28 12:32
      → C:\Users\Administrator\Untitled5.ipynb
   4. Untitled6.ipynb                                0.3 KB   2026-09-28 09:39
      → C:\Users\Administrator\Untitled6.ipynb
   5. 01_phase1_repodb_discovery.ipynb             315.9 KB   2026-09-25 11:22
      → C:\Users\Administrator\01_phase1_repodb_discovery.ipynb
   6. EPMO HUB.ipynb                               447.3 KB   2026-09-19 12:34
      → C:\Users\Administrator\EPMO HUB.ipynb
   7. Untitled4.ipynb                              364.8 KB   2026-09-18 22:30
      → C:\Users\Administrator\Untitled4.ipynb
   8. Untitled3.ipynb                               16.7 KB   2026-09-18 19:19


In [45]:
import json
from pathlib import Path

home = Path(r"C:\Users\Administrator")

candidates = [
    "Untitled8.ipynb",
    "Untitled7.ipynb",
    "Untitled5.ipynb",
    "Untitled6.ipynb",
    "01_phase1_repodb_discovery.ipynb",
]

for name in candidates:
    p = home / name
    if not p.exists():
        print(f"\n❌ Not found: {name}")
        continue
    
    try:
        data = json.loads(p.read_text(encoding="utf-8"))
        cells = data.get("cells", [])
        
        # First 3 non-empty code cells
        code_cells = []
        for c in cells:
            if c.get("cell_type") == "code":
                src = "".join(c.get("source", [])).strip()
                if src:
                    code_cells.append(src.split("\n")[0][:70])
            if len(code_cells) >= 3:
                break
        
        # Check for RepurposeAlpha markers
        all_source = " ".join(
            "".join(c.get("source", [])) for c in cells
        )
        is_our_project = any(marker in all_source for marker in [
            "RepurposeAlpha", "PROJECT_ROOT", "repodb", "correlation", 
            "streamlit", "markowitz", "chEMBL", "OPTIONS"
        ])
        
        print(f"\n{'=' * 70}")
        print(f"📓 {name}")
        print(f"{'=' * 70}")
        print(f"   Total cells: {len(cells)}")
        print(f"   RepurposeAlpha-related: {'✅ YES' if is_our_project else '❌ no markers found'}")
        print(f"   First code lines:")
        for line in code_cells:
            print(f"      {line}")
    except Exception as e:
        print(f"\n❌ {name} — error: {e}")


📓 Untitled8.ipynb
   Total cells: 44
   RepurposeAlpha-related: ✅ YES
   First code lines:
      from pathlib import Path
      %%writefile C:\Users\Administrator\RepurposeAlpha\src\executive\__init
      %%writefile C:\Users\Administrator\RepurposeAlpha\app\app.py

📓 Untitled7.ipynb
   Total cells: 7
   RepurposeAlpha-related: ✅ YES
   First code lines:
      from pathlib import Path
      %%writefile C:\Users\Administrator\RepurposeAlpha\docs\how_it_works.ht
      import subprocess

📓 Untitled5.ipynb
   Total cells: 156
   RepurposeAlpha-related: ✅ YES
   First code lines:
      # New notebook: setup
      # Cell 49: Prepare RepoDB training data
      # Cell 50: Feature engineering — drug history, disease history, phase

📓 Untitled6.ipynb
   Total cells: 1
   RepurposeAlpha-related: ❌ no markers found
   First code lines:

📓 01_phase1_repodb_discovery.ipynb
   Total cells: 86
   RepurposeAlpha-related: ✅ YES
   First code lines:
      # Cell 1: Install packages
      # Cell 01: Inst